# 03a_contig_id_gene_context.ipynb — The deep dive
This notebook zooms back in on the most interesting signal that emerged from notebook 02: a clade of cyanobacteria with no close SILVA matches, detected in multiple geographically separated samples. It is a targeted investigation of those sequences specifically.

Prodigal predicts protein-coding genes from the cyano contigs, and those proteins are sent to remote BLASTp against NCBI nr to characterize the genomic content. A gene arrow map then places each contig's annotated genes in linear context, centered on the 16S locus, giving a visual sense of the genomic neighborhood around the rRNA gene. Finally, a pairwise 16S identity matrix is computed across all novel sequences — applying the Yarza et al. (2014) thresholds (94.5% genus, 98.7% species) — to assess whether the sequences from different lakes represent the same species, the same genus, or something more divergent. A companion interpretation table lays out the ecological and taxonomic implications of each pairwise comparison. The conclusion is that at least some of these sequences represent a genuinely novel genus present in geographically separated freshwater environments.

## Contig Identity & Gene Context — KY-Mam-100624-C and MN-Col-091924-B

> **Resolved: these contigs are green algal chloroplasts.** See FLAG 4 in `README.md`.
>
> This notebook exists *because* plastid contamination was the identified risk for this
> dataset. It gathers the contigs whose 16S landed in SILVA's chloroplast bin with no named
> host — carried under the working tag `'Novel cyanobacterium'`, meaning "identity
> unresolved" — and then §3 settles them: flanking-gene BLASTp returns psbC, atpF/atpH and
> plastid LAGLIDADG intron ORFs at 92–99% identity to green algal chloroplast genomes. The
> phylogeny in `03b_16S_phylogeny.ipynb` puts all six inside the Chlorophyta plastid
> radiation. §3 is not a caveat on this notebook — it is the notebook's result.

**Formerly** "Novel Cyanobacteria Deep Dive". The rename reflects what it does: establish
contig identity from gene context, for the candidates flagged in
`02_all_samples_community.ipynb`.
**Notebook 03** — Focused analysis of novel cyanobacterial lineages identified in `02_all_samples_community.ipynb`

## Priority samples
| Sample | Location | Cyano depth | Notes |
|---|---|---|---|
| MN-Col-091924-B | Columbia Lake, MN | 168x | Highest-confidence; unknown genus |
| KY-Mam-100624-C | Mammoth Cave, KY (interior) | 7x | Most novel; cave environment; unknown genus |

## Goals
1. Rule out chloroplast origin using genomic context (flanking genes on contig)
2. Confirm novelty — compare to NCBI nr, not just SILVA
3. Characterize the contigs: length, GC%, completeness
4. Extract and annotate protein-coding genes on novel cyano contigs
5. Look for taxonomically informative marker genes (rpoB, rpoC, recA)
6. Build a more targeted phylogeny using additional reference genomes


In [1]:
# ── 0. Setup ─────────────────────────────────────────────────────────────────
from pathlib import Path
import subprocess, json, sys
import pandas as pd
import numpy as np

# Bootstrap: find project root to load shared utilities
_p = Path.cwd().resolve()
for _parent in [_p] + list(_p.parents):
    if (_parent / 'Snakefile').exists():
        sys.path.insert(0, str(_parent / 'notebooks'))
        break
from utils import find_project_root, load_config, is_placeholder

PROJECT   = find_project_root()
cfg       = load_config()
TAX_DIR   = PROJECT / 'results' / '10_tax_profile'
POLISH_DIR= PROJECT / 'results' / '04_polish'
GENE_DIR  = PROJECT / 'results' / '11_genes'
COV_DIR   = PROJECT / 'results' / '06_cov'
FIG_DIR   = PROJECT / 'results' / 'figures'
OUT_DIR   = PROJECT / 'results' / '13_contig_id'
OUT_DIR.mkdir(exist_ok=True)

QUERY_SAMPLES = ['KY-Mam-100624-C', 'MN-Col-091924-B']

SAMPLE_LABELS = {
    'KY-Mam-100624-C': 'Mammoth Cave, KY\n(interior)',
    'MN-Col-091924-B': 'Columbia Lake, MN',
}

print('Output directory:', OUT_DIR)

Output directory: /Users/jaemac/Desktop/Palmer_Lab/Cyanobacteria/GenomicAnalyses/MetagenomicAnalyses/metagenome/results/13_novel_cyano


In [2]:
import shutil

# ── Tool availability check — run before any other cells ─────────────────────
_required = {
    'blastn':    'BLAST+   (conda: blast)',
    'barrnap':   'barrnap  (conda: barrnap)',
    'prodigal':  'Prodigal (conda: prodigal)',
    'hmmsearch': 'HMMER    (conda: hmmer)',
    'minimap2':  'minimap2 (conda: minimap2)',
    'samtools':  'samtools (conda: samtools)',
}
_missing = {tool: pkg for tool, pkg in _required.items() if not shutil.which(tool)}
if _missing:
    raise EnvironmentError(
        "Missing tools — activate the correct environment before running:\n" +
        "\n".join(f"  {tool:10s}  {pkg}" for tool, pkg in _missing.items())
    )
print("Tool check passed:", list(_required))

Tool check passed: ['blastn', 'barrnap', 'prodigal', 'hmmsearch', 'minimap2', 'samtools']


## 1. Identify candidate contigs from SILVA's chloroplast bin

Extract the contigs whose 16S top hit falls in `Bacteria;Cyanobacteriota;...;Chloroplast`
with a placeholder at the deepest rank (`uncultured` / `metagenome` / `Incertae Sedis`),
i.e. SILVA's chloroplast bin with **no named host alga**.

> **These are chloroplasts** — see FLAG 4 in `README.md`.
> This selection was always a *to-resolve* list: SILVA files plastid 16S under
> Cyanobacteriota (plastids descend from cyanobacteria by primary endosymbiosis), so a hit
> in that bin with no named host means the host alga is unidentified, not that the sequence
> is novel. Section 3 below (flanking-gene BLASTp) and the phylogeny in
> `03b_16S_phylogeny.ipynb` were built to answer it, and did. The selection is retained
> unchanged because it defines exactly the contig set that evidence covers — what changed
> is that the label now states the answer instead of deferring it.


In [ ]:
# ── 1. Pull candidate contigs from SILVA's chloroplast bin ───────────────────
import re, sys
# is_placeholder imported from utils (via cell 0 setup)

def is_unnamed_plastid_bin(stitle):
    """True for SILVA hits in Bacteria;Cyanobacteriota;...;Chloroplast whose deepest
    rank is a placeholder — SILVA's chloroplast bin with no named host alga.

    A placeholder here means the HOST is unidentified in SILVA, NOT that the sequence
    is a novel free-living cyanobacterium. Sections 3 (flanking-gene BLASTp) and
    03b_16S_phylogeny.ipynb (16S phylogeny) resolved every one of these as a green algal
    chloroplast — see FLAG 4 in README.md.

    Previously named is_novel_cyano(); the selection logic is unchanged.
    """
    if stitle.startswith('Eukaryota'):
        return False
    if 'Cyanobacteriota' not in stitle or 'Chloroplast' not in stitle:
        return False
    taxa = stitle.split(' ', 1)[-1].split(';')
    deep = taxa[-1].strip()
    # Named host alga → plastid with a known host (handled separately)
    if deep and not is_placeholder(deep) and deep.lower() != 'incertae sedis':
        return False
    return True

plastid_bin_contigs = {}   # sample → list of {contig, qid, pct_id, align_len, stitle}

for s in QUERY_SAMPLES:
    blast_f = TAX_DIR / s / 'blast_16S_silva.json'
    if not blast_f.exists():
        print(f'{s}: no BLAST file'); continue
    blast = json.load(open(blast_f))
    hits = []
    for qid, hit_list in blast.items():
        if not hit_list: continue
        top = hit_list[0]
        if is_unnamed_plastid_bin(top['title']):
            contig = qid.split('::')[1].split(':')[0] if '::' in qid else qid
            hits.append({
                'qid':       qid,
                'contig':    contig,
                'pct_id':    top['pct_id'],
                'align_len': top.get('align_len', 0),
                'stitle':    top['title'][:80],
            })
    plastid_bin_contigs[s] = hits
    print(f'\n{s}: {len(hits)} chloroplast-bin 16S sequences')
    for h in hits:
        print(f"  {h['contig']:30s}  {h['pct_id']:.1f}%  aln={h['align_len']}bp  {h['stitle']}")


## 2. Contig characterisation
For each candidate contig: length, GC content, self-coverage depth.


In [4]:
# ── 2. Contig stats: length, GC%, coverage depth ─────────────────────────────
from Bio import SeqIO

def gc(seq):
    """Return GC content (%) for a sequence string or Bio.Seq object."""
    s = str(seq).upper()
    return (s.count('G') + s.count('C')) / len(s) * 100 if s else 0

rows = []
for s, hits in plastid_bin_contigs.items():
    assembly = POLISH_DIR / f'{s}.racon2.fasta'
    depth_f  = COV_DIR / s / 'depth.tsv'

    # Load assembly contigs
    contigs = {r.id: r for r in SeqIO.parse(assembly, 'fasta')}

    # Load depth
    dep = {}
    if depth_f.exists():
        dep_df = pd.read_csv(depth_f, sep='\t')
        bam_cols = [c for c in dep_df.columns if c.endswith('.bam')]
        dcol = bam_cols[0] if bam_cols else dep_df.columns[1]
        dep = dict(zip(dep_df.iloc[:, 0], dep_df[dcol]))

    seen = set()
    for h in hits:
        c = h['contig']
        if c in seen: continue
        seen.add(c)
        rec = contigs.get(c)
        rows.append({
            'sample':  s,
            'contig':  c,
            'length':  len(rec.seq) if rec else None,
            'gc_pct':  round(gc(rec.seq), 1) if rec else None,
            'depth':   round(dep.get(c, float('nan')), 1),
            'pct_id':  h['pct_id'],
        })

df_contigs = pd.DataFrame(rows)
print(df_contigs.to_string(index=False))

         sample      contig  length  gc_pct  depth  pct_id
KY-Mam-100624-C  contig_225   46945    33.6    8.5  93.631
KY-Mam-100624-C  contig_227   72723    35.2    9.0  93.631
KY-Mam-100624-C  contig_531   16909    34.2    6.5  96.981
KY-Mam-100624-C  contig_427   22464    27.9    3.1  96.628
MN-Col-091924-B contig_1504   94827    29.5  159.7  95.283
MN-Col-091924-B contig_3677   71888    30.2  174.6  97.459


In [ ]:
# ──── 2c. Genome recovery vs. a fixed cyanobacterial benchmark ──────────────────────────
# Completeness = total assembled bp / assumed genome size.
#
# The 2–6 Mb Oscillatoriales range is a DELIBERATE FIXED BENCHMARK, applied uniformly to
# every sample so the numbers are comparable to each other. At the time this ran, whether
# these contigs were cyanobacteria was the open question — assuming a cyanobacterial genome
# size was the hypothesis under test, not an assumption of the answer.
#
# That question is now settled: these are green algal chloroplast genomes (FLAG 4 in
# README.md). The percentages below are unchanged and still valid as a cross-sample
# yardstick, but see the second reading in the notes at the bottom before quoting them.

from Bio import SeqIO

GENOME_SIZE_MIN = 2e6   # smallest known Oscillatoriales genome (~2 Mb)
GENOME_SIZE_TYP = 6e6   # typical filamentous cyanobacterium (~5-7 Mb)

print(f"{'Sample':<25} {'Contigs':>8} {'Total bp':>12} {'% (2 Mb genome)':>17} {'% (6 Mb genome)':>17}")
print('-' * 82)

for s, hits in plastid_bin_contigs.items():
    assembly = POLISH_DIR / f'{s}.racon2.fasta'
    if not assembly.exists():
        print(f'{s}: assembly not found')
        continue

    asm = {r.id: len(r.seq) for r in SeqIO.parse(assembly, 'fasta')}
    # hits is a list of dicts, each with a 'contig' key
    contig_ids = list({h['contig'] for h in hits})   # deduplicate
    lengths    = [asm[c] for c in contig_ids if c in asm]
    total_bp   = sum(lengths)

    pct_min = total_bp / GENOME_SIZE_MIN * 100
    pct_typ = total_bp / GENOME_SIZE_TYP * 100

    print(f'{s:<25} {len(lengths):>8,} {total_bp:>12,} {pct_min:>16.1f}% {pct_typ:>16.1f}%')
    for cid in contig_ids:
        l = asm.get(cid, 0)
        print(f'  {cid:<23} {l:>12,} bp')

print(f"""
Notes — benchmark reading (as run):
  - Fixed benchmark: Oscillatoriales genome {GENOME_SIZE_MIN/1e6:.0f}–{GENOME_SIZE_TYP/1e6:.0f} Mb,
    applied uniformly to every sample so recovery is comparable across them
  - Both samples recover ~160 kb — comparable coverage
  - At typical genome size, recovery is ~2–3%; at minimum, up to ~8%
  - At this level, absence of any given gene (e.g. KaiABC) cannot be confirmed

Notes — second reading (identity now resolved, see FLAG 4 in README.md):
  - These contigs are green algal CHLOROPLAST genomes, not cyanobacterial chromosomes
  - Green algal plastid genomes run 103–204 kb (Mychonastes jurisii 103 kb,
    Chlorella vulgaris 151 kb, Scenedesmus obliquus 161 kb, Chlamydomonas reinhardtii 204 kb)
  - Against that denominator the SAME assembled lengths are ~65–76% of a complete
    organelle genome — substantially complete, not a 2–3% fragment
  - The gene-absence caveat above therefore does NOT apply to KaiABC: plastids do not
    carry kaiABC at all (lost during endosymbiotic genome reduction), so its absence is
    expected regardless of recovery depth
""")


In [ ]:
# ────── 2d. Genome completeness – per strain, constrained to tree taxa ──────────────────────────
#
# NORMALIZATION NOTE (see FLAG 4 in README.md) — the 'Novel' entries are retained on purpose.
# Every strain here is normalized against a CYANOBACTERIAL genome size, which was the right
# common yardstick when the identity of these contigs was still the open question. The
# entries labelled 'Novel' are now known to be green algal CHLOROPLAST genomes, so their
# bars are an artifact of that normalization: a ~160 kb plastid genome divided by a 5–7 Mb
# cyanobacterial denominator reads as ~2–3% when it is in fact a substantially complete
# organelle genome (green algal plastids run 103–204 kb). The genuine cyanobacteria in this
# figure are unaffected and read normally. Kept as-is so the cross-strain comparison stays
# on one scale; do not quote the 'Novel' percentages as genome completeness.
#
# Only shows strains that appear in the final phylogenetic tree (notebook 02).
# Tree taxa are read from all_samples_cyano_16S.fna produced by cyano-extract.
# For each tree taxon, completeness uses ONLY the single representative contig
# from the tree (the longest 16S-bearing contig per taxon per sample).
# Novel lineages (KY-Mam-C and MN-Col) use 16S-anchored override (all listed contigs).
# SILVA_OVERRIDE corrects known Kraken2 misclassifications at the tree-contig level.

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import re
from Bio import SeqIO
from collections import defaultdict

# ────── Step 1: Parse tree FASTA → {sample: set of contig_ids} ───────────────────────────────────
# FASTA ID format: {sample}|16S_rRNA__{contig}_{start}-{end}_{strand}
tree_fna = TAX_DIR / 'all_samples_cyano_16S.fna'
if not tree_fna.exists():
    raise FileNotFoundError(
        f'{tree_fna} not found — run cyano-extract cell in notebook 02 first.')

tree_contig_map = {}   # sample → set of contig_ids that appear in the tree
for rec in SeqIO.parse(tree_fna, 'fasta'):
    sample  = rec.id.split('|')[0]
    payload = rec.id.split('|', 1)[1]
    parts   = payload.split('__', 1)
    m = re.match(r'(contig_\d+)_\d+-\d+_[+-]', parts[1]) if len(parts) > 1 else None
    contig  = m.group(1) if m else parts[-1]
    tree_contig_map.setdefault(sample, set()).add(contig)

print('Samples / contigs represented in the phylogenetic tree:')
for s, ctgs in sorted(tree_contig_map.items()):
    print(f'  {s}: {sorted(ctgs)}')

# ────── Genome size range (Mb) by lineage ─────────────────────────────────────────────────────────
GENOME_SIZES = {
    'Synechococcales':   (2.0, 3.5),
    'Synechococcus':     (2.0, 3.5),
    'Cyanobium':         (2.0, 3.0),
    'Limnotrichales':    (4.5, 6.0),
    'Limnothrix':        (4.5, 6.0),
    'Halomicronema':     (3.5, 5.0),
    'Phormidesmiales':   (4.0, 5.5),
    'Nostocales':        (6.0, 8.5),
    'Anabaena':          (5.5, 7.5),
    'Oscillatoriales':   (5.0, 7.0),
    'Novel':             (5.0, 7.0),
    'default':           (2.0, 8.0),
}

def genome_size_range(taxon_str):
    for key, rng in GENOME_SIZES.items():
        if key in taxon_str:
            return rng
    return GENOME_SIZES['default']

# ────── SILVA override: correct known Kraken2 misclassifications ──────────────────────────────────
# Key: (sample, contig) → correct label based on SILVA BLAST
# TN-Cla contig_35: Kraken2 says "Synechococcus sp." but SILVA says Limnotrichaceae at 99.7%
#   — confirmed by two 16S copies on the same 4.6 Mb contig (consistent with Limnothrix genome)
SILVA_OVERRIDE = {
    ('TN-Cla-092324-E', 'contig_35'): 'Limnothrix sp.',
}

# ────── Step 2: Kraken2 helpers ───────────────────────────────────────────────────────────────────
def get_cyano_taxids(report_path):
    df_r = pd.read_csv(report_path, sep='\t', header=None,
                       names=['pct','clade','direct','rank','taxid','name'])
    taxids, in_cyano, cyano_indent = set(), False, None
    for _, row in df_r.iterrows():
        name   = str(row['name'])
        indent = len(name) - len(name.lstrip())
        name   = name.strip()
        if name in ('Cyanobacteria', 'Cyanobacteriota', 'Cyanobacteria/Melainabacteria group'):
            in_cyano, cyano_indent = True, indent
        if in_cyano:
            if indent <= cyano_indent and name not in ('Cyanobacteria','Cyanobacteriota','Cyanobacteria/Melainabacteria group'):
                in_cyano = False
            else:
                taxids.add(int(row['taxid']))
    return taxids

def kraken_contig_label(tsv_path, cyano_taxids, target_contigs):
    result = {}
    with open(tsv_path) as f:
        for line in f:
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 3 or parts[0] != 'C':
                continue
            contig, taxon = parts[1], parts[2]
            if contig not in target_contigs:
                continue
            m = re.search(r'taxid (\d+)', taxon)
            if not m or int(m.group(1)) not in cyano_taxids:
                continue
            label = re.sub(r'\s*\(taxid \d+\)', '', taxon).strip()
            result[contig] = ' '.join(label.split()[:2])
    return result

# ────── Step 3: 16S-anchored override for the chloroplast-bin contigs ───────────────────────────
PLASTID_OVERRIDE = {
    'KY-Mam-100624-C': {'Chloroplast, green algal (KY)': ['contig_225','contig_227','contig_531','contig_427']},
    'MN-Col-091924-B': {'Chloroplast, green algal (MN)': ['contig_1504','contig_3677']},
}

# ────── Step 4: Calculate completeness — one entry per tree contig ────────────────────────────────
rows = []

for s, tree_ctgs in sorted(tree_contig_map.items()):
    report   = TAX_DIR / s / 'contigs.kraken.report'
    tsv      = TAX_DIR / s / 'contigs.kraken.tsv'
    assembly = POLISH_DIR / f'{s}.racon2.fasta'

    if not all(p.exists() for p in [report, tsv, assembly]):
        print(f'{s}: missing files — skip')
        continue

    asm_lens     = {r.id: len(r.seq) for r in SeqIO.parse(assembly, 'fasta')}
    cyano_taxids = get_cyano_taxids(report)
    contig_label = kraken_contig_label(tsv, cyano_taxids, tree_ctgs)

    # Apply SILVA override (corrects Kraken2 misclassifications)
    for (ovr_sample, ovr_contig), ovr_label in SILVA_OVERRIDE.items():
        if ovr_sample == s and ovr_contig in tree_ctgs:
            contig_label[ovr_contig] = ovr_label
            print(f'  SILVA override: {s} {ovr_contig} → {ovr_label}')

    # Novel override: group all override contigs together
    plastid_ctgs_done = set()
    if s in PLASTID_OVERRIDE:
        for label, ctgs in PLASTID_OVERRIDE[s].items():
            plastid_ctgs_done.update(ctgs)
            total_bp = sum(asm_lens.get(c, 0) for c in ctgs)
            if total_bp == 0:
                continue
            lo, hi = genome_size_range(label)
            rows.append({'sample': s, 'strain': label, 'n_contigs': len(ctgs),
                         'total_bp': total_bp,
                         'pct_lo': total_bp / (hi * 1e6) * 100,
                         'pct_hi': total_bp / (lo * 1e6) * 100,
                         'pct_mid': total_bp / ((lo + hi) / 2 * 1e6) * 100})

    for tc in sorted(tree_ctgs):
        if tc in plastid_ctgs_done:
            continue
        length = asm_lens.get(tc, 0)
        if length == 0:
            continue
        strain = contig_label.get(tc, tc)
        lo, hi = genome_size_range(strain)
        rows.append({'sample': s, 'strain': strain, 'n_contigs': 1,
                     'total_bp': length,
                     'pct_lo': length / (hi * 1e6) * 100,
                     'pct_hi': length / (lo * 1e6) * 100,
                     'pct_mid': length / ((lo + hi) / 2 * 1e6) * 100})

df = pd.DataFrame(rows).sort_values(['sample','total_bp'], ascending=[True, False])
print(f'\n{len(df)} strains to plot')
print(df[['sample','strain','n_contigs','total_bp','pct_lo','pct_hi']].to_string(index=False))

# ────── Step 5: Presentation figure ──────────────────────────────────────────────────────────────
BG, AX_BG   = '#2b2b2b', '#333333'
PLASTID_COLOR = "#E8C24E"
KNOWN_COLOR = "#09CC51"

n = len(df)
labels = [f"{r['strain']}\n({r['sample'].split('-')[0]}-{r['sample'].split('-')[1]})"
          for _, r in df.iterrows()]

with plt.style.context('dark_background'):
    fig, ax = plt.subplots(figsize=(10, max(5, n * 0.6 + 1.5)), facecolor=BG)
    ax.set_facecolor(AX_BG)

    for i, (_, r) in enumerate(df.iterrows()):
        # Keyed to the override labels above — these are the plastid entries whose
        # bars are a normalization artifact (see note at the top of this cell).
        is_plastid = 'Chloroplast' in r['strain']
        color    = PLASTID_COLOR if is_plastid else KNOWN_COLOR
        plot_lo  = min(r['pct_lo'], 100.0)
        plot_hi  = min(r['pct_hi'], 100.0)
        plot_mid = min(r['pct_mid'], 100.0)
        capped   = r['pct_hi'] > 100.0

        ax.barh(i, plot_hi - plot_lo, left=plot_lo,
                height=0.5, color=color, alpha=0.35, zorder=2,
                hatch='///' if capped else None)
        ax.scatter(plot_mid, i, color=color, s=60, zorder=3,
                   marker='>' if capped else 'o')
        ax.text(plot_hi + 0.5, i,
                f"{r['pct_lo']:.1f}–{r['pct_hi']:.1f}%  ({r['total_bp']/1e3:.0f} kb, {r['n_contigs']} contig{'s' if r['n_contigs']>1 else ''})"
                + (' ▶' if capped else ''),
                va='center', fontsize=7.5, color='#cccccc')

    ax.set_yticks(range(n))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel('Estimated genome completeness (%)', fontsize=10)
    ax.set_title('Genome recovery by strain, normalized to cyanobacterial genome size\n(taxa reported in phylogenetic tree)',
                 fontsize=11, fontweight='bold')
    ax.set_xlim(0, 130)
    ax.axvline(100, color='#aaaaaa', lw=0.8, ls=':')
    ax.text(100.5, -0.6, '100%', fontsize=7, color='#aaaaaa')
    ax.spines[['top','right']].set_visible(False)
    for sp in ax.spines.values(): sp.set_edgecolor('#555')
    ax.axvline(1, color='#888', lw=0.8, ls='--')
    ax.text(1.3, n - 0.5, '1%', fontsize=7, color='#888')

    handles = [
        mpatches.Patch(color=PLASTID_COLOR, alpha=0.8, label='Chloroplast — normalization artifact, see note'),
        mpatches.Patch(color=KNOWN_COLOR, alpha=0.8, label='Known lineage (Kraken2 classified)'),
        mpatches.Patch(facecolor='none', edgecolor='grey', hatch='///', label='Exceeds reference range (▶ = capped)'),
    ]
    ax.legend(handles=handles, fontsize=8, loc='lower right',
              facecolor='#3a3a3a', edgecolor='#555', framealpha=0.9)

    fig.text(0.5, -0.03,
             'Bar = estimated range (min–max genome size for lineage).  '
             'Hatched bars exceed reference range — likely near-complete genome or multiple strains.  '
             'Tree representative contig only, except the chloroplast entries (4 contigs KY / 2 contigs MN).\n'
             'All bars normalized to a CYANOBACTERIAL genome size. The chloroplast entries are green algal '
             'plastid genomes (103-204 kb), so their low values are an artifact of that shared denominator, '
             'not low recovery - see FLAG 4 in README.md.',
             ha='center', fontsize=7, color='#888888', wrap=True)

    plt.tight_layout()
    fig.savefig(FIG_DIR / 'completeness_by_strain.pdf', bbox_inches='tight', dpi=200, facecolor=BG)
    fig.savefig(FIG_DIR / 'completeness_by_strain.png', bbox_inches='tight', dpi=200, facecolor=BG)
    plt.show()

# ────── Save table ────────────────────────────────────────────────────────────────────────────────
tables_dir = PROJECT / 'results' / 'tables'
tables_dir.mkdir(exist_ok=True)
df.to_csv(tables_dir / 'completeness_by_strain.csv', index=False)
print(f'\nSaved → {FIG_DIR}/completeness_by_strain.pdf')
print(f'Saved → {tables_dir}/completeness_by_strain.csv')


In [7]:
# ── 2b. Run Prodigal on any sample missing gene predictions ──────────────────
import shutil

for s in QUERY_SAMPLES:
    gene_dir = GENE_DIR / s
    gff = gene_dir / 'all_genes.gff'
    if gff.exists():
        print(f'{s}: Prodigal already done')
        continue

    assembly = POLISH_DIR / f'{s}.racon2.fasta'
    if not assembly.exists():
        print(f'{s}: assembly not found, skipping')
        continue

    if not shutil.which('prodigal'):
        print('prodigal not found — install: conda install -n genomics_arm64 prodigal')
        break

    gene_dir.mkdir(parents=True, exist_ok=True)
    print(f'{s}: running Prodigal...')
    subprocess.run([
        'prodigal',
        '-i', str(assembly),
        '-a', str(gene_dir / 'all_genes.faa'),
        '-d', str(gene_dir / 'all_genes.fna'),
        '-f', 'gff',
        '-o', str(gff),
        '-p', 'meta',       # metagenome mode
        '-q',               # quiet
    ], check=True)
    print(f'{s}: done → {gff}')


KY-Mam-100624-C: Prodigal already done
MN-Col-091924-B: Prodigal already done


## 3. Genomic context — rule out chloroplast origin
Extract flanking genes on each candidate contig using Prodigal annotations. This is the test that resolved their identity.
If flanking genes are eukaryotic in origin → likely chloroplast.
If flanking genes are bacterial → free-living cyanobacterium.


In [8]:
# ── 3. Genomic context: genes flanking each novel 16S locus ─────────────────
# For each novel cyano contig, show:
#   - Total predicted genes
#   - Genes within ±10 kb of the 16S locus
#   - Gene IDs for BLASTp follow-up

import re

FLANK_BP = 10000  # window around 16S gene to inspect

for s, hits in plastid_bin_contigs.items():
    gff = GENE_DIR / s / 'all_genes.gff'
    if not gff.exists():
        print(f'{s}: no GFF — run Prodigal cell first'); continue

    # Parse GFF into dict: contig → list of (start, end, strand, gene_id)
    contig_genes = {}
    for line in open(gff):
        if line.startswith('#'): continue
        p = line.strip().split('\t')
        if len(p) < 9: continue
        c, start, end, strand = p[0], int(p[3]), int(p[4]), p[6]
        # Build FAA-compatible ID: contig_NAME_NUM (extract num from ID=X_NUM)
        gid_raw = re.search(r'ID=([^;]+)', p[8])
        if gid_raw:
            num = gid_raw.group(1).split('_')[-1]
            gid = f'{c}_{num}'
        else:
            gid = '?'
        contig_genes.setdefault(c, []).append((start, end, strand, gid))

    print(f'\n{"="*65}')
    print(f'  {s}')
    print(f'{"="*65}')

    seen = set()
    for h in hits:
        c = h['contig']
        if c in seen: continue
        seen.add(c)

        # 16S coordinates from barrnap qid: contig::contig_X:start-end(strand)
        m = re.search(r':(\d+)-(\d+)\(([+-])\)', h['qid'])
        if m:
            s16_start, s16_end = int(m.group(1)), int(m.group(2))
        else:
            s16_start, s16_end = 0, 0

        all_genes = contig_genes.get(c, [])
        flanking  = [(st, en, strand, gid) for st, en, strand, gid in all_genes
                     if abs(st - s16_start) <= FLANK_BP or abs(en - s16_end) <= FLANK_BP]

        print(f'\n  Contig: {c}  |  16S: {s16_start}–{s16_end}  |  '
              f'Total genes: {len(all_genes)}  |  Within ±{FLANK_BP//1000}kb: {len(flanking)}')
        print(f'  {"Gene ID":<25} {"Start":>8} {"End":>8} {"Strand":>7}  Dist to 16S')
        print(f'  {"-"*60}')
        for st, en, strand, gid in sorted(flanking, key=lambda x: x[0]):
            dist = min(abs(st - s16_start), abs(en - s16_end),
                       abs(st - s16_end),   abs(en - s16_start))
            marker = ' ◀ 16S' if abs(st - s16_start) < 50 else ''
            print(f'  {gid:<25} {st:>8} {en:>8} {strand:>7}  {dist:>8} bp{marker}')
        if not flanking:
            print('  (no genes in flanking window)')



  KY-Mam-100624-C

  Contig: contig_225  |  16S: 39095–40636  |  Total genes: 18  |  Within ±10kb: 9
  Gene ID                      Start      End  Strand  Dist to 16S
  ------------------------------------------------------------
  contig_225_10                22600    33081       +      6014 bp
  contig_225_11                33881    34129       +      4966 bp
  contig_225_12                34633    35283       +      3812 bp
  contig_225_13                35938    36927       +      2168 bp
  contig_225_14                38174    38509       +       586 bp
  contig_225_15                41883    42278       -      1247 bp
  contig_225_16                42414    42650       +      1778 bp
  contig_225_17                44777    45496       +      4141 bp
  contig_225_18                46792    46944       -      6156 bp

  Contig: contig_227  |  16S: 71083–72624  |  Total genes: 49  |  Within ±10kb: 8
  Gene ID                      Start      End  Strand  Dist to 16S
  -------------

In [9]:
# ── 4. Extract flanking proteins + run BLASTp to confirm bacterial origin ────
# Proteins within 2 kb of each 16S locus are BLASTed against NCBI nr.
# Results are cached to disk — re-running the cell loads existing results.

from Bio import SeqIO
import re, subprocess, shutil

CLOSE_BP = 2000  # extract genes within this distance of the 16S locus

for s, hits in plastid_bin_contigs.items():
    faa_all = GENE_DIR / s / 'all_genes.faa'
    gff     = GENE_DIR / s / 'all_genes.gff'
    if not faa_all.exists() or not gff.exists():
        print(f'{s}: missing Prodigal output — run cell 6 first'); continue

    # ── Load all predicted proteins ──────────────────────────────────────────
    proteins = {r.id.split()[0]: str(r.seq).rstrip('*')
                for r in SeqIO.parse(faa_all, 'fasta')}

    # ── Parse GFF ────────────────────────────────────────────────────────────
    contig_genes = {}
    for line in open(gff):
        if line.startswith('#'): continue
        p = line.strip().split('\t')
        if len(p) < 9: continue
        ctg, start, end, strand = p[0], int(p[3]), int(p[4]), p[6]
        gid_raw = re.search(r'ID=([^;]+)', p[8])
        if gid_raw:
            num = gid_raw.group(1).split('_')[-1]
            gid = f'{ctg}_{num}'
        else:
            gid = '?'
        contig_genes.setdefault(ctg, []).append((start, end, strand, gid))

    out_faa   = OUT_DIR / f'{s}_flanking_proteins.faa'
    blast_out = OUT_DIR / f'{s}_flanking_blastp.tsv'

    # ── Extract flanking proteins ─────────────────────────────────────────────
    written = 0
    with open(out_faa, 'w') as fh:
        seen = set()
        for h in hits:
            c = h['contig']
            if c in seen: continue
            seen.add(c)
            m = re.search(r':(\d+)-(\d+)\(([+-])\)', h['qid'])
            s16_start = int(m.group(1)) if m else 0
            s16_end   = int(m.group(2)) if m else 0
            for st, en, strand, gid in sorted(contig_genes.get(c, []), key=lambda x: x[0]):
                dist = min(abs(st - s16_start), abs(en - s16_end),
                           abs(st - s16_end),   abs(en - s16_start))
                if dist <= CLOSE_BP and gid in proteins:
                    seq = proteins[gid]
                    fh.write(f'>{gid} contig={c} dist_to_16S={dist}bp strand={strand}\n{seq}\n')
                    print(f'  {gid:<28} dist={dist:>5} bp  len={len(seq)} aa')
                    written += 1

    print(f'\n{s}: {written} flanking proteins written to {out_faa}')

    # ── Run BLASTp against NCBI nr ────────────────────────────────────────────
    if blast_out.exists():
        print(f'  BLASTp results cached — loading {blast_out.name}')
    else:
        if not shutil.which('blastp'):
            print('  blastp not found — install: conda install -n genomics_arm64 blast')
            continue
        print(f'  Running blastp -remote vs NCBI nr (may take 2–5 min per sample)...')
        res = subprocess.run(
            [
                'blastp', '-query', str(out_faa),
                '-db', 'nr', '-remote',
                '-outfmt', '6 qseqid stitle pident evalue bitscore',
                '-max_target_seqs', '3',
                '-out', str(blast_out),
            ],
            capture_output=True, text=True,
        )
        if res.returncode != 0:
            print(f'  BLASTp FAILED (exit {res.returncode}):\n{res.stderr[:500]}')
            continue
        print(f'  Done → {blast_out}')

    # ── Display top hit per query ─────────────────────────────────────────────
    if blast_out.exists() and blast_out.stat().st_size > 0:
        cols = ['qseqid', 'stitle', 'pident', 'evalue', 'bitscore']
        bt = pd.read_csv(blast_out, sep='\t', names=cols)
        top = (bt.sort_values('bitscore', ascending=False)
                 .groupby('qseqid').first().reset_index())
        top['stitle'] = top['stitle'].str[:65]
        print(f'\n  Top BLASTp hits (top match per protein):')
        print(top[['qseqid', 'pident', 'evalue', 'stitle']].to_string(index=False))
        print()
        # Interpretation hint
        bact_hits = top[top['stitle'].str.contains(r'bacterium|cyanobacter|prokaryot', case=False, na=False)]
        euk_hits  = top[top['stitle'].str.contains(r'chloroplast|plant|algae|virid|embryophyt', case=False, na=False)]
        print(f'  Bacterial-like hits: {len(bact_hits)}  |  Plastid/eukaryotic hits: {len(euk_hits)}')
        if euk_hits.empty:
            print('  ✓ No plastid/eukaryotic hits — consistent with free-living cyanobacterium')
        else:
            print('  ⚠ Some hits suggest eukaryotic/plastid origin — review manually')
    else:
        print(f'  No BLASTp results found in {blast_out}')


  contig_225_14                dist=  586 bp  len=111 aa
  contig_225_15                dist= 1247 bp  len=131 aa
  contig_225_16                dist= 1778 bp  len=78 aa
  contig_227_48                dist= 1777 bp  len=78 aa
  contig_227_49                dist= 1246 bp  len=131 aa
  contig_531_1                 dist=  165 bp  len=142 aa
  contig_531_2                 dist=  707 bp  len=154 aa
  contig_427_10                dist= 1611 bp  len=222 aa
  contig_427_11                dist=  745 bp  len=82 aa

KY-Mam-100624-C: 9 flanking proteins written to /Users/jaemac/Desktop/Palmer_Lab/Cyanobacteria/GenomicAnalyses/MetagenomicAnalyses/metagenome/results/13_novel_cyano/KY-Mam-100624-C_flanking_proteins.faa
  BLASTp results cached — loading KY-Mam-100624-C_flanking_blastp.tsv

  Top BLASTp hits (top match per protein):
       qseqid  pident        evalue                                                            stitle
contig_225_14  96.364  8.550000e-73 photosystem II 44 kDa protein (chl

### 3b. Convert NCBI BLAST download → notebook format
Reads the NCBI BLAST hit table (downloaded from web), fetches organism descriptions
via Entrez, and overwrites the per-sample TSV files used by the figure cell.

In [10]:
# ── Convert NCBI BLAST alignment download → notebook-expected TSV format ───────
from Bio import Entrez
import time

ALIGNMENT_FILE = PROJECT / 'data' / 'manual' / 'WB7RKP8C014-Alignment.txt'

# Map contig prefix → sample
CONTIG_TO_SAMPLE = {
    'contig_225': 'KY-Mam-100624-C',
    'contig_227': 'KY-Mam-100624-C',
    'contig_531': 'KY-Mam-100624-C',
    'contig_427': 'KY-Mam-100624-C',
    'contig_1504': 'MN-Col-091924-B',
    'contig_3677': 'MN-Col-091924-B',
}

Entrez.email = cfg.get('entrez_email')  # required by NCBI

# ── 1. Parse: top hit per query (first data row = highest bitscore) ────────────
top_hits = {}  # qseqid → {sacc, pident, evalue, bitscore}
current_query = None
with open(ALIGNMENT_FILE) as fh:
    for line in fh:
        line = line.rstrip()
        if line.startswith('# Query:'):
            current_query = line.split('# Query: ')[1].split()[0]
        elif line and not line.startswith('#') and current_query:
            if current_query not in top_hits:
                parts = line.split('\t')
                top_hits[current_query] = dict(
                    sacc=parts[1], pident=float(parts[2]),
                    evalue=float(parts[10]), bitscore=float(parts[11])
                )

print(f'Queries with hits: {len(top_hits)}')
for qid, h in top_hits.items():
    print(f'  {qid:<28}  top hit: {h["sacc"]}  ({h["pident"]:.1f}%  e={h["evalue"]})')

# ── 2. Fetch organism descriptions from Entrez ────────────────────────────────
unique_accs = list(set(v['sacc'] for v in top_hits.values()))
print(f'\nFetching descriptions for {len(unique_accs)} unique accessions via Entrez...')

acc_to_title = {}
BATCH = 10
for i in range(0, len(unique_accs), BATCH):
    batch = unique_accs[i:i+BATCH]
    try:
        handle = Entrez.esummary(db='protein', id=','.join(batch))
        records = Entrez.read(handle)
        handle.close()
        for rec in records:
            acc = rec.get('AccessionVersion', '')
            title = rec.get('Title', '')
            org   = rec.get('Organism', '')
            acc_to_title[acc] = f'{title} [{org}]'
    except Exception as e:
        print(f'  Entrez error for batch {i//BATCH}: {e}')
        for a in batch:
            acc_to_title.setdefault(a, a)
    time.sleep(0.4)  # NCBI rate limit

# ── 3. Write per-sample TSV files (overwrites empty files from cell above) ────
sample_rows = {s: [] for s in QUERY_SAMPLES}
for qid, hit in top_hits.items():
    ctg_base = '_'.join(qid.split('_')[:2])
    sample = CONTIG_TO_SAMPLE.get(ctg_base)
    if sample:
        stitle = acc_to_title.get(hit['sacc'], hit['sacc'])
        sample_rows[sample].append(
            (qid, stitle, hit['pident'], hit['evalue'], hit['bitscore'])
        )

print()
for s, rows in sample_rows.items():
    out_path = OUT_DIR / f'{s}_flanking_blastp.tsv'
    with open(out_path, 'w') as fh:
        for r in rows:
            fh.write('\t'.join(str(x) for x in r) + '\n')
    print(f'{s}: {len(rows)} hits written → {out_path.name}')
    for r in rows:
        print(f'  {r[0]:<28}  {r[1][:80]}')

Queries with hits: 10
  contig_225_14                 top hit: YP_010564914.1  (96.4%  e=8.55e-73)
  contig_225_15                 top hit: CAL6436292.1  (55.6%  e=6.29e-24)
  contig_227_49                 top hit: CAL6436292.1  (55.6%  e=6.29e-24)
  contig_531_1                  top hit: CAP2980638.1  (55.2%  e=9.87e-11)
  contig_531_2                  top hit: KAJ4757151.1  (55.9%  e=6.37e-27)
  contig_427_10                 top hit: YP_010488900.1  (92.1%  e=2.1e-114)
  contig_427_11                 top hit: YP_009185204.1  (98.8%  e=7.42e-47)
  contig_1504_61                top hit: NP_045800.2  (92.8%  e=0.0)
  contig_1504_62                top hit: YP_009629475.1  (63.3%  e=1.81e-94)
  contig_3677_50                top hit: YP_009629475.1  (63.3%  e=1.81e-94)

Fetching descriptions for 8 unique accessions via Entrez...

KY-Mam-100624-C: 7 hits written → KY-Mam-100624-C_flanking_blastp.tsv
  contig_225_14                 photosystem II 44 kDa protein (chloroplast) [Chlamydomonas c

In [11]:
# ── 3c. Non-cyanobacterial flanking hits — inspection table ─────────────────
# Shows eukaryotic/plastid and other-bacterial top hits (matching the figure's
# color scheme). Click any accession to open its NCBI protein page.
import re
import pandas as pd
from IPython.display import display, HTML

# ── Classification — mirrors classify_hit() in the gene arrow map cell ────────
CYANO_KEYWORDS = [
    'cyanobacter', 'synechococcus', 'synechocystis', 'prochlorococcus',
    'anabaena', 'nostoc', 'microcystis', 'aphanizomenon', 'planktothrix',
    'oscillatoria', 'limnothrix', 'cyanobium', 'gloeobacter', 'chroococcus',
    'spirulina', 'arthrospira', 'lyngbya', 'trichodesmium', 'acaryochloris',
    'thermosynechococcus', 'crocosphaera', 'raphidiopsis', 'cylindrospermum',
    'fischerella', 'calothrix', 'rivularia', 'nodularia', 'loriellopsis',
    'halomicronema', 'nodosilinea', 'tychonema',
]
EUK_KEYWORDS = ['chloroplast', 'plant', 'alga', 'virid', 'embryophyt']

def classify_hit(stitle, organism):
    t = (str(stitle) + ' ' + str(organism)).lower()
    if not t.strip():
        return 'No database hit'
    if any(k in t for k in CYANO_KEYWORDS):
        return 'Cyanobacterial'
    if any(k in t for k in EUK_KEYWORDS):
        return 'Eukaryotic/plastid'
    return 'Other bacterial'

# ── Re-parse ALIGNMENT_FILE to recover accessions (not stored in TSV) ─────────
top_hits_raw = {}
current_query = None
with open(ALIGNMENT_FILE) as fh:
    for line in fh:
        line = line.rstrip()
        if line.startswith('# Query:'):
            current_query = line.split('# Query: ')[1].split()[0]
        elif line and not line.startswith('#') and current_query:
            if current_query not in top_hits_raw:
                parts = line.split('\t')
                top_hits_raw[current_query] = parts[1]   # sacc only

# ── Load per-sample TSVs ───────────────────────────────────────────────────────
all_rows = []
for s in QUERY_SAMPLES:
    tsv = OUT_DIR / f'{s}_flanking_blastp.tsv'
    if not tsv.exists():
        print(f'Missing: {tsv.name} — run cell 14 first'); continue
    with open(tsv) as fh:
        for line in fh:
            parts = line.rstrip().split('\t')
            qid, stitle = parts[0], parts[1]
            sacc = top_hits_raw.get(qid, '—')
            org_m = re.search(r'\[([^\[\]]+)\]', stitle)
            organism = org_m.group(1) if org_m else '—'
            desc = re.sub(r'\s*\[[^\]]*\](\s*\[\])?$', '', stitle).strip()
            all_rows.append({
                'Sample':      s,
                'Protein':     qid,
                'Contig':      '_'.join(qid.split('_')[:2]),
                'Description': desc,
                'Organism':    organism,
                'Accession':   sacc,
                '% ID':        float(parts[2]),
                'E-value':     float(parts[3]),
                'Bitscore':    float(parts[4]),
                'Category':    classify_hit(stitle, organism),
            })

df = pd.DataFrame(all_rows)

# ── Filter to non-cyanobacterial hits ─────────────────────────────────────────
show = df[df['Category'] != 'Cyanobacterial'].copy().reset_index(drop=True)
n_cyano = (df['Category'] == 'Cyanobacterial').sum()
print(f"{len(show)} non-cyanobacterial hits shown  ({n_cyano} cyanobacterial excluded)\n")

# ── Build HTML table with clickable NCBI links ────────────────────────────────
CAT_STYLE = {
    'Eukaryotic/plastid': 'background:#6A1E8C;color:white',
    'Other bacterial':    'background:#B84010;color:white',
}
NCBI_URL = 'https://www.ncbi.nlm.nih.gov/protein/{}'

rows_html = []
for _, r in show.sort_values(['Category', 'Sample', 'Protein']).iterrows():
    style = CAT_STYLE.get(r['Category'], '')
    link  = (f'<a href="{NCBI_URL.format(r["Accession"])}" target="_blank" '
             f'style="color:inherit">{r["Accession"]}</a>')
    rows_html.append(
        f'<tr style="{style}">'
        f'<td style="padding:4px 8px">{r["Sample"]}</td>'
        f'<td style="padding:4px 8px">{r["Protein"]}</td>'
        f'<td style="padding:4px 8px">{r["Description"]}</td>'
        f'<td style="padding:4px 8px"><i>{r["Organism"]}</i></td>'
        f'<td style="padding:4px 8px">{link}</td>'
        f'<td style="padding:4px 8px;text-align:right">{r["% ID"]:.1f}%</td>'
        f'<td style="padding:4px 8px;text-align:right">{r["E-value"]:.2e}</td>'
        f'<td style="padding:4px 8px;text-align:right">{r["Bitscore"]:.0f}</td>'
        f'<td style="padding:4px 8px">{r["Category"]}</td>'
        f'</tr>'
    )

header = (
    '<tr style="background:#222;color:white;font-size:12px">'
    '<th style="padding:5px 8px">Sample</th>'
    '<th style="padding:5px 8px">Protein</th>'
    '<th style="padding:5px 8px">Description</th>'
    '<th style="padding:5px 8px">Organism (top hit)</th>'
    '<th style="padding:5px 8px">Accession</th>'
    '<th style="padding:5px 8px">% ID</th>'
    '<th style="padding:5px 8px">E-value</th>'
    '<th style="padding:5px 8px">Bitscore</th>'
    '<th style="padding:5px 8px">Category</th>'
    '</tr>'
)

html = (
    '<table style="border-collapse:collapse;font-size:13px;width:100%">'
    + header + ''.join(rows_html) + '</table>'
)
display(HTML(html))


10 non-cyanobacterial hits shown  (0 cyanobacterial excluded)



Sample,Protein,Description,Organism (top hit),Accession,% ID,E-value,Bitscore,Category
KY-Mam-100624-C,contig_225_14,photosystem II 44 kDa protein (chloroplast),Chlamydomonas chlamydogama,YP_010564914.1,96.4%,8.55e-73,224,Eukaryotic/plastid
KY-Mam-100624-C,contig_427_10,ATP synthase CF0 B subunit (chloroplast),Desmodesmus abundans,YP_010488900.1,92.1%,2.10e-114,334,Eukaryotic/plastid
KY-Mam-100624-C,contig_427_11,CF0 subunit III of ATP synthase (chloroplast),Bracteacoccus giganteus,YP_009185204.1,98.8%,7.42e-47,155,Eukaryotic/plastid
MN-Col-091924-B,contig_1504_61,photosystem II 44 kDa protein (chloroplast),Chlorella vulgaris,NP_045800.2,92.8%,0.00e+00,863,Eukaryotic/plastid
MN-Col-091924-B,contig_1504_62,putative LAGLIDADG homing endonuclease (chloroplast),Coelastrella saipanensis,YP_009629475.1,63.3%,1.81e-94,286,Eukaryotic/plastid
MN-Col-091924-B,contig_3677_50,putative LAGLIDADG homing endonuclease (chloroplast),Coelastrella saipanensis,YP_009629475.1,63.3%,1.81e-94,286,Eukaryotic/plastid
KY-Mam-100624-C,contig_225_15,unnamed protein product,Bathycoccus prasinos,CAL6436292.1,55.6%,6.29e-24,101,Other bacterial
KY-Mam-100624-C,contig_227_49,unnamed protein product,Bathycoccus prasinos,CAL6436292.1,55.6%,6.29e-24,101,Other bacterial
KY-Mam-100624-C,contig_531_1,unnamed protein product,Hordeum erectifolium,CAP2980638.1,55.2%,9.87e-11,65,Other bacterial
KY-Mam-100624-C,contig_531_2,ORF44l,Rhynchospora pubera,KAJ4757151.1,55.9%,6.37e-27,107,Other bacterial


## 4. Next steps
- [ ] BLASTp flanking proteins → confirm bacterial vs eukaryotic context
- [ ] Extract and BLASTn novel 16S sequences against NCBI nt (not just SILVA)
- [ ] Identify additional marker genes (rpoB, rpoC, recA) on novel cyano contigs
- [ ] Build targeted phylogeny with NCBI reference cyanobacteria genomes
- [ ] Compare KY-Mam-C vs MN-Col-B novel sequences — same genus?


In [ ]:
# ── 5. Gene arrow map — genomic context of novel 16S loci ────────────────────
# Each contig is drawn centered on its 16S locus (x = 0).
# Gene colors reflect BLASTp category vs NCBI nr.
# The bottom row is S. elongatus PCC 7942 (reference genome) as a positive
# control — genes are classified from GenBank annotation (authoritative, no
# NCBI timeout risk). All flanking genes should show as Cyanobacterial.

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import urllib.request
import re

WINDOW_BP = 12000   # show ±12 kb around each 16S locus

# ── BLASTp category colors ────────────────────────────────────────────────────
CAT_COLORS = {
    'Cyanobacterial':    '#2196A6',
    'Other bacterial':   '#E8824E',
    'No database hit':   '#5C85D6',
    'Eukaryotic/plastid':'#9C4FC4',
    '16S rRNA':          '#F5C518',
}

# Comprehensive list of cyanobacterial genera/terms for hit classification
CYANO_KEYWORDS = [
    'cyanobacter', 'synechococcus', 'synechocystis', 'prochlorococcus',
    'anabaena', 'nostoc', 'microcystis', 'aphanizomenon', 'planktothrix',
    'oscillatoria', 'limnothrix', 'cyanobium', 'gloeobacter', 'chroococcus',
    'spirulina', 'arthrospira', 'lyngbya', 'trichodesmium', 'acaryochloris',
    'thermosynechococcus', 'crocosphaera', 'raphidiopsis', 'cylindrospermum',
    'fischerella', 'calothrix', 'rivularia', 'nodularia', 'loriellopsis',
    'halomicronema', 'nodosilinea', 'tychonema',
]

def classify_hit(stitle):
    """Assign a BLASTp hit title to one of: Cyanobacterial, Eukaryotic/plastid,
    Other bacterial, or No database hit.
    Classification is keyword-based on the hit description string.
    """
    if pd.isna(stitle) or not str(stitle).strip():
        return 'No database hit'
    s = str(stitle).lower()
    if any(x in s for x in CYANO_KEYWORDS):
        return 'Cyanobacterial'
    if any(x in s for x in ['chloroplast', 'plant', 'alga', 'virid', 'embryophyt']):
        return 'Eukaryotic/plastid'
    return 'Other bacterial'

def parse_gff(gff_path):
    """Parse a Prodigal GFF file and return a dict of {contig: [(start, end, strand, gene_id)]}.
    gene_id is built as '{contig}_{num}' to match the FAA header format.
    """
    genes = {}
    for line in open(gff_path):
        if line.startswith('#'): continue
        p = line.strip().split('\t')
        if len(p) < 9: continue
        ctg = p[0]
        start, end, strand = int(p[3]), int(p[4]), p[6]
        gid_m = re.search(r'ID=([^;]+)', p[8])
        if gid_m:
            num = gid_m.group(1).split('_')[-1]
            gid = f'{ctg}_{num}'
        else:
            gid = '?'
        genes.setdefault(ctg, []).append((start, end, strand, gid))
    return genes

# ── Load BLASTp results for novel samples ─────────────────────────────────────
blast_hits = {}
n_blastp_loaded = 0
blastp_ran = False

for s in QUERY_SAMPLES:
    blast_out = OUT_DIR / f'{s}_flanking_blastp.tsv'
    if blast_out.exists():
        blastp_ran = True
        if blast_out.stat().st_size > 0:
            bt = pd.read_csv(blast_out, sep='\t',
                             names=['qseqid','stitle','pident','evalue','bitscore'])
            top = bt.sort_values('bitscore', ascending=False).groupby('qseqid').first()
            for gid, row in top.iterrows():
                blast_hits[gid] = classify_hit(row['stitle'])
                n_blastp_loaded += 1

if n_blastp_loaded:
    status = f'{n_blastp_loaded} BLASTp hits loaded'
elif blastp_ran:
    status = 'BLASTp complete — no NCBI nr hits returned for this set'
else:
    status = 'BLASTp not yet run — re-run cell 9 first'
print(status)

# ── Reference genome: S. elongatus PCC 7942 (positive control) ───────────────
# Uses GenBank annotation directly — avoids remote BLASTp timeouts and provides
# the most authoritative gene classifications for a fully annotated genome.
REF_DIR       = PROJECT / 'results' / 'ref'
REF_DIR.mkdir(exist_ok=True)
ref_gbk       = REF_DIR / 'S_elongatus_PCC7942.gbk'
REF_ACCESSION = 'CP000100.1'
REF_LABEL     = 'S. elongatus PCC 7942\n(GenBank annotation — positive control)'

# 1. Download GenBank annotation if not cached
if not ref_gbk.exists():
    print(f'Downloading {REF_ACCESSION} annotation...')
    url = (f'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi'
           f'?db=nuccore&id={REF_ACCESSION}&rettype=gb&retmode=text')
    urllib.request.urlretrieve(url, ref_gbk)
    print(f'  → {ref_gbk.name}  ({ref_gbk.stat().st_size/1e6:.1f} Mb)')
else:
    print(f'Reference annotation cached: {ref_gbk.name}')

# 2. Parse GenBank → find 16S locus + CDS features in window
ref_row = None
if ref_gbk.exists():
    from Bio import SeqIO as _SeqIO
    ref_rec  = next(_SeqIO.parse(ref_gbk, 'genbank'))

    # Find first 16S rRNA feature
    s16_start = s16_end = None
    for feat in ref_rec.features:
        if feat.type == 'rRNA':
            prod = ' '.join(feat.qualifiers.get('product', []))
            if '16S' in prod:
                s16_start = int(feat.location.start)
                s16_end   = int(feat.location.end)
                break

    if s16_start is not None:
        s16_mid = (s16_start + s16_end) / 2
        print(f'  16S rRNA at {s16_start}–{s16_end} ({s16_end-s16_start} bp)')

        # Collect CDS features within window
        nearby_ref = []
        ref_blast_hits = {}
        feat_id = 0
        for feat in ref_rec.features:
            if feat.type != 'CDS':
                continue
            fstart = int(feat.location.start)
            fend   = int(feat.location.end)
            fmid   = (fstart + fend) / 2
            if abs(fmid - s16_mid) > WINDOW_BP:
                continue
            strand = '+' if feat.location.strand >= 0 else '-'
            gid    = f'ref_cds_{feat_id}'
            feat_id += 1
            nearby_ref.append((fstart, fend, strand, gid))
            # Classify from annotation: all PCC 7942 CDS are cyanobacterial
            ref_blast_hits[gid] = 'Cyanobacterial'

        print(f'  {len(nearby_ref)} CDS features in ±{WINDOW_BP//1000} kb window')

        ref_row = dict(
            sample='ref', contig=REF_LABEL,
            s16_start=s16_start, s16_end=s16_end, s16_mid=s16_mid,
            genes=nearby_ref, pct_id=100.0,
            blast_override=ref_blast_hits,
        )
    else:
        print('  No 16S rRNA feature found in GenBank record')

# ── Parse GFF + collect plot rows (novel samples) ─────────────────────────────
plot_rows = []
for s, hits in plastid_bin_contigs.items():
    gff = GENE_DIR / s / 'all_genes.gff'
    if not gff.exists():
        print(f'{s}: no GFF'); continue
    contig_genes = parse_gff(gff)

    seen = set()
    for h in hits:
        c = h['contig']
        if c in seen: continue
        seen.add(c)
        m = re.search(r':(\d+)-(\d+)\(([+-])\)', h['qid'])
        if not m: continue
        s16_start, s16_end = int(m.group(1)), int(m.group(2))
        s16_mid = (s16_start + s16_end) / 2

        nearby = [(st, en, strand, gid)
                  for st, en, strand, gid in contig_genes.get(c, [])
                  if abs((st + en) / 2 - s16_mid) <= WINDOW_BP]

        plot_rows.append(dict(
            sample=s, contig=c,
            s16_start=s16_start, s16_end=s16_end, s16_mid=s16_mid,
            genes=nearby, pct_id=h['pct_id'],
            blast_override=None,
        ))

if ref_row:
    plot_rows.append(ref_row)
else:
    print('Reference row not available — check GenBank download')

# ── Draw ──────────────────────────────────────────────────────────────────────
def gene_arrow(ax, x_start, x_end, strand, color, y=0, h=0.55):
    width  = x_end - x_start
    head_l = min(abs(width) * 0.28, 350)
    if strand == '+':
        ax.add_patch(mpatches.FancyArrow(
            x_start, y, width, 0,
            width=h, head_width=h * 1.55, head_length=head_l,
            length_includes_head=True,
            fc=color, ec='white', lw=0.35, zorder=2))
    else:
        ax.add_patch(mpatches.FancyArrow(
            x_end, y, -width, 0,
            width=h, head_width=h * 1.55, head_length=head_l,
            length_includes_head=True,
            fc=color, ec='white', lw=0.35, zorder=2))

n = len(plot_rows)
fig, axes = plt.subplots(n, 1, figsize=(14, 1.0 * n + 1.2), squeeze=False)
fig.subplots_adjust(hspace=0.15)

ARROW_H = 0.25
GENE_Y  = 0.0

for i, (ax, row) in enumerate(zip(axes[:, 0], plot_rows)):
    s16_mid   = row['s16_mid']
    s16_start = row['s16_start']
    s16_end   = row['s16_end']
    hits_src  = row['blast_override'] if row['blast_override'] is not None else blast_hits
    is_ref    = row['sample'] == 'ref'

    if is_ref:
        ax.set_facecolor('#FFFBE6')
        ax.axhline(ARROW_H * 2.8, color='#CCCCCC', lw=0.8, ls='--')

    ax.axhline(GENE_Y, color='#AAAAAA', lw=1.1, zorder=0)

    for st, en, strand, gid in row['genes']:
        cat   = hits_src.get(gid, 'No database hit')
        color = CAT_COLORS[cat]
        gene_arrow(ax, st - s16_mid, en - s16_mid, strand, color,
                   y=GENE_Y, h=ARROW_H)

    s16_w = s16_end - s16_start
    gene_arrow(ax, -s16_w / 2, s16_w / 2, '+', CAT_COLORS['16S rRNA'],
               y=GENE_Y, h=ARROW_H * 1.35)
    ax.text(0, ARROW_H * 1.25, '16S', ha='center', va='bottom',
            fontsize=7.5, color='#333', fontweight='bold', zorder=4)

    if is_ref:
        row_label = row['contig']
    else:
        sample_lbl = SAMPLE_LABELS.get(row['sample'], row['sample'])
        row_label  = f"{sample_lbl}\n{row['contig']}\n({row['pct_id']:.1f}% 16S id)"

    ax.set_ylabel(row_label, fontsize=7.5, rotation=0,
                  ha='right', va='center', labelpad=8)

    ax.set_xlim(-WINDOW_BP, WINDOW_BP)
    ax.set_ylim(-ARROW_H * 2.8, ARROW_H * 3.2)
    ax.set_yticks([])
    ax.spines[['top', 'right', 'left']].set_visible(False)

    if i < n - 1:
        ax.set_xticks([])
        ax.spines['bottom'].set_visible(False)
    else:
        ax.xaxis.set_major_locator(mticker.MultipleLocator(3000))
        ax.xaxis.set_major_formatter(
            mticker.FuncFormatter(
                lambda x, _: f'{int(x/1000):+d} kb' if x != 0 else '0'))
        ax.set_xlabel('Distance from 16S rRNA locus (bp)', fontsize=9)

patches = [mpatches.Patch(color=v, label=k) for k, v in CAT_COLORS.items()]
axes[0, 0].legend(handles=patches, loc='upper right', fontsize=8,
                  framealpha=0.92, ncol=1, handlelength=1.5)
                  

fig.suptitle(
    'Genomic context of the 16S rRNA loci — chloroplast identification\n'
    f'Query samples: BLASTp vs NCBI nr  —  {status}\n'
    'Bottom row: S. elongatus PCC 7942 (GenBank annotation, positive control)',
    fontsize=11, fontweight='bold')

out_fig = FIG_DIR / 'contig_gene_map.pdf'
fig.savefig(out_fig, bbox_inches='tight', dpi=200)
fig.savefig(str(out_fig).replace('.pdf', '.png'), bbox_inches='tight', dpi=200)
plt.show()
print(f'Saved → {out_fig}')

In [ ]:
# ── 6. Extract & align novel 16S sequences — are the cave contigs distinct? ──
# Extracts 16S sequences using barrnap coordinates, aligns with MAFFT,
# and computes a pairwise identity matrix to confirm distinct lineages.
# Also reports SILVA alignment coverage (qcov) per sequence as a QC check:
# low qcov (<70%) means only a partial 16S was compared, weakening novelty claims.

import subprocess, shutil
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from Bio import SeqIO, AlignIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

out_16S = OUT_DIR / 'query_16S.fna'
out_aln = OUT_DIR / 'query_16S.aln.fna'

# ── Extract 16S sequences ────────────────────────────────────────────────────
# Deduplication: keep longest hit per contig; discard partials < MIN_LEN
MIN_LEN = 1000  # bp — full 16S ~1400-1550 bp

records  = []
seq_meta = {}   # seq_id → {length, align_len, qcov}

for s, hits in plastid_bin_contigs.items():
    assembly = POLISH_DIR / f'{s}.racon2.fasta'
    contigs  = {r.id: r for r in SeqIO.parse(assembly, 'fasta')}
    short_s  = s.replace('KY-Mam-100624-C', 'KY-Mam-C').replace('MN-Col-091924-B', 'MN-Col-B')

    # contig_name → (seq_str, pct_id, length, align_len)
    best_per_contig = {}
    for h in hits:
        m = re.search(r':(\d+)-(\d+)\(([+-])\)', h['qid'])
        if not m: continue
        start, end, strand = int(m.group(1)), int(m.group(2)), m.group(3)
        rec = contigs.get(h['contig'])
        if not rec: continue

        seq = rec.seq[start:end]
        if strand == '-':
            seq = seq.reverse_complement()
        seq_str   = str(seq).upper()
        align_len = h.get('align_len', 0)

        prev = best_per_contig.get(h['contig'])
        if prev is None or len(seq_str) > prev[2]:
            best_per_contig[h['contig']] = (seq_str, h['pct_id'], len(seq_str), align_len)

    for contig, (seq_str, pct_id, length, align_len) in best_per_contig.items():
        if length < MIN_LEN:
            print(f"  SKIPPED {short_s}|{contig}  len={length} bp (< {MIN_LEN} bp minimum)")
            continue
        qcov   = align_len / length * 100 if length > 0 else 0
        seq_id = f"{short_s}|{contig}|{pct_id:.1f}%"
        records.append(SeqRecord(Seq(seq_str), id=seq_id, description=''))
        seq_meta[seq_id] = {'length': length, 'align_len': align_len, 'qcov': qcov}
        flag = '  ⚠ low qcov' if qcov < 70 else ''
        print(f"  {seq_id}  len={length} bp  aln={align_len} bp  qcov={qcov:.0f}%{flag}")

SeqIO.write(records, out_16S, 'fasta')
print(f'\n{len(records)} sequences written to {out_16S.name}')

# ── SILVA alignment coverage QC table ────────────────────────────────────────
print('\n── SILVA alignment coverage (qcov) ──────────────────────────────────────')
print(f"  {'Sequence':<35} {'Seq len':>8} {'Aln len':>8} {'qcov':>7}  {'Status'}")
print(f"  {'-'*35} {'-------':>8} {'-------':>8} {'------':>7}  -------")
all_ok = True
for sid, m in seq_meta.items():
    status = 'OK' if m['qcov'] >= 70 else '⚠ PARTIAL — identity estimate less certain'
    if m['qcov'] < 70: all_ok = False
    print(f"  {sid:<35} {m['length']:>8} {m['align_len']:>8} {m['qcov']:>6.0f}%  {status}")
if all_ok:
    print('\n  All sequences have ≥70% query coverage — identity estimates are reliable.')
else:
    print('\n  ⚠ Some sequences have low query coverage. Identity may reflect only partial gene.')
print('  (Full 16S = ~1400-1550 bp; qcov ≥ 90% is ideal; ≥ 70% is acceptable)')

# ── Align with MAFFT ──────────────────────────────────────────────────────────
if not shutil.which('mafft'):
    print('mafft not found — install: conda install -n genomics_arm64 mafft')
else:
    print('\nRunning MAFFT alignment...')
    res = subprocess.run(
        ['mafft', '--auto', '--quiet', str(out_16S)],
        capture_output=True, text=True,
    )
    if res.returncode != 0:
        print(f'MAFFT error:\n{res.stderr[:400]}')
    else:
        out_aln.write_text(res.stdout)
        print(f'Alignment written to {out_aln.name}')

# ── Pairwise identity matrix ──────────────────────────────────────────────────
if not out_aln.exists():
    print('No alignment found — check MAFFT step above')
else:
    aln   = AlignIO.read(out_aln, 'fasta')
    n_seq = len(aln)
    ids   = [r.id for r in aln]
    seqs  = [str(r.seq).upper() for r in aln]

    def pairwise_pid(s1, s2):
        """Identity over shared aligned region — skips columns where either seq has a gap."""
        match = total = 0
        for a, b in zip(s1, s2):
            if a == '-' or b == '-': continue
            if a == b: match += 1
            total += 1
        return match / total * 100 if total else 0

    mat = np.zeros((n_seq, n_seq))
    for i in range(n_seq):
        for j in range(n_seq):
            mat[i, j] = pairwise_pid(seqs[i], seqs[j])

    labels = [i.replace('KY-Mam-C|', 'KY: ').replace('MN-Col-B|', 'MN: ') for i in ids]

    # ── Print table ───────────────────────────────────────────────────────────
    print('\nPairwise 16S identity matrix (%):\n')
    col_w = max(len(l) for l in labels) + 2
    print(' ' * col_w + ''.join(f'{l:>{col_w}}' for l in labels))
    for i, lbl in enumerate(labels):
        print(f'{lbl:<{col_w}}' + ''.join(f'{mat[i,j]:>{col_w}.1f}' for j in range(n_seq)))

    # ── Heatmap ───────────────────────────────────────────────────────────────
    off_diag = mat[~np.eye(n_seq, dtype=bool)]
    vmin = np.floor(off_diag.min())

    cmap = mcolors.LinearSegmentedColormap.from_list('white_red', ['white', '#000000'])

    cell_sz = 1.3
    fig, ax = plt.subplots(figsize=(n_seq * cell_sz + 2.5, n_seq * cell_sz + 1.5))

    mask     = np.triu(np.ones_like(mat, dtype=bool), k=1)
    mat_plot = np.ma.masked_where(mask, mat)
    cmap.set_bad('white')
    im   = ax.imshow(mat_plot, vmin=vmin, vmax=100, cmap=cmap, aspect='auto')
    cbar = plt.colorbar(im, ax=ax, label='% 16S identity', shrink=0.6, pad=0.02)

    ax.set_xticks(range(n_seq))
    ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=12)
    ax.set_yticks(range(n_seq))
    ax.set_yticklabels(labels, fontsize=12)

    for i in range(n_seq):
        for j in range(n_seq):
            if j > i: continue
            mid = (vmin + 100) / 2
            txt_color = 'black' if mat[i, j] < mid else 'white'
            ax.text(j, i, f'{mat[i,j]:.1f}', ha='center', va='center',
                    fontsize=12, color=txt_color,
                    fontweight='bold' if i == j else 'normal')

    ax.set_title('Pairwise 16S rRNA identity — chloroplast contigs',
                 fontsize=14, fontweight='bold', pad=28)
    ax.text(0.5, 1.02, 'Genus (94.5%) and species (98.7%) demarcation thresholds (Yarza et al. 2014)',
            transform=ax.transAxes, ha='center', va='bottom', fontsize=11, color='#555555')

    plt.tight_layout()
    fig_out = FIG_DIR / 'contig_16S_identity.pdf'
    fig.savefig(fig_out, bbox_inches='tight', dpi=200)
    fig.savefig(str(fig_out).replace('.pdf', '.png'), bbox_inches='tight', dpi=200)
    plt.show()
    print(f'\nSaved → {fig_out}  (colour range: {vmin:.0f}–100%)')

    # ── Interpretation table ──────────────────────────────────────────────────
    GENUS_THRESH   = 94.5
    SPECIES_THRESH = 98.7

    table_rows = []
    for i in range(n_seq):
        for j in range(i):
            pid = mat[i, j]
            if pid >= SPECIES_THRESH:
                interp = 'Same species'
            elif pid >= GENUS_THRESH:
                interp = 'Same genus, different species'
            else:
                interp = 'Different genera'
            table_rows.append([labels[j], labels[i], f'{pid:.1f}%', interp])

    col_headers = ['Sequence A', 'Sequence B', '16S identity', 'Interpretation']
    col_widths  = [0.22, 0.22, 0.12, 0.34]

    fig2, ax2 = plt.subplots(figsize=(11, 0.45 * len(table_rows) + 1.2))
    ax2.axis('off')

    tbl = ax2.table(
        cellText=table_rows,
        colLabels=col_headers,
        colWidths=col_widths,
        bbox=[0, 0, 1, 1],
        cellLoc='left',
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1, 1.6)

    for j in range(len(col_headers)):
        tbl[0, j].set_facecolor('#2c2c2c')
        tbl[0, j].set_text_props(color='white', fontweight='bold')

    ROW_COLORS = {
        'Same species':                 '#fde8e8',
        'Same genus, different species':'#fff4cc',
        'Different genera':             '#e8f4e8',
    }
    for r, row in enumerate(table_rows, start=1):
        bg = ROW_COLORS.get(row[3], 'white')
        for j in range(len(col_headers)):
            tbl[r, j].set_facecolor(bg)
            tbl[r, j].set_edgecolor('#cccccc')

    ax2.set_title('Pairwise 16S identity — interpretations\n'
                  f'Genus threshold: {GENUS_THRESH}%  |  Species threshold: {SPECIES_THRESH}%  '
                  '(Yarza et al. 2014)',
                  fontsize=11, fontweight='bold', pad=6)

    fig2.tight_layout()
    tbl_out = FIG_DIR / 'contig_16S_identity_table.pdf'
    fig2.savefig(tbl_out, bbox_inches='tight', dpi=200)
    fig2.savefig(str(tbl_out).replace('.pdf', '.png'), bbox_inches='tight', dpi=200)
    plt.show()
    print(f'Saved → {tbl_out}')


## 7. Pan-sample KaiABC Screen

Search all 10 assembled metagenomes for circadian clock genes (KaiA, KaiB, KaiC) using HMMER against `KaiABC_correct.hmm`. Prodigal is run first for any sample missing protein predictions.

**HMM profiles used** (`db/kai_hmm/KaiABC_correct.hmm`):

| Profile | Gene | Source |
|---------|------|--------|
| KaiA | KaiA C-terminal domain | PF07688 (Pfam) |
| KaiA_N | KaiA N-terminal domain | PF21714 (Pfam) |
| circ_KaiB | KaiB | TIGR02654 (JCVI) |
| circ_KaiC | KaiC | TIGR02655 (JCVI) |
| KaiB_longform_Anabaena7120 | KaiB (long-form) | Custom HMM — *Anabaena* PCC 7120 seed (added 2026-04) |


In [14]:
# ── 7. Pan-sample KaiABC screen ───────────────────────────────────────────────
import shutil, csv

ALL_SAMPLES = [
    'KY-Mam-100624-B', 'KY-Mam-100624-C',
    'MN-Por-091024-A', 'TN-Cla-092324-E',
    'WI-Bar-090824-B', 'NC-Che-092424-E',
    'GA-Cor-092624-B', 'FL-Wim-100124-B',
    'MN-Col-091924-B', 'FL-Hom-052325-G',
]

HMM_DB   = PROJECT / 'db' / 'kai_hmm' / 'KaiABC_correct.hmm'
KAI_DIR  = PROJECT / 'results' / '14_kai_screen'
KAI_DIR.mkdir(exist_ok=True)

EVALUE   = 1e-5   # same threshold used in 00_driver_2.0

# ── Step 1: Prodigal for any sample missing gene predictions ─────────────────
print('=== Step 1: Prodigal gene prediction ===')
for s in ALL_SAMPLES:
    gene_dir = GENE_DIR / s
    gff      = gene_dir / 'all_genes.gff'
    faa      = gene_dir / 'all_genes.faa'
    if faa.exists():
        n = sum(1 for l in open(faa) if l.startswith('>'))
        print(f'  {s}: already done ({n:,} proteins)')
        continue
    assembly = POLISH_DIR / f'{s}.racon2.fasta'
    if not assembly.exists():
        print(f'  {s}: assembly not found — skipping')
        continue
    gene_dir.mkdir(parents=True, exist_ok=True)
    print(f'  {s}: running Prodigal...')
    cmd = [
        'prodigal',
        '-i', str(assembly),
        '-a', str(faa),
        '-d', str(gene_dir / 'all_genes.fna'),
        '-f', 'gff', '-o', str(gff),
        '-p', 'meta', '-q',
    ]
    print('  CMD:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
    n = sum(1 for l in open(faa) if l.startswith('>'))
    print(f'  {s}: done — {n:,} proteins')

# ── Step 2: hmmsearch all samples ────────────────────────────────────────────
print('\n=== Step 2: hmmsearch vs KaiABC_correct.hmm ===')
results = {}   # sample -> list of hit dicts

for s in ALL_SAMPLES:
    faa     = GENE_DIR / s / 'all_genes.faa'
    tblout  = KAI_DIR / f'{s}_kai.tblout'

    if not faa.exists():
        print(f'  {s}: no FAA — skipped')
        results[s] = []
        continue

    if not tblout.exists():
        cmd = [
            'hmmsearch',
            '--tblout', str(tblout),
            '-E',       str(EVALUE),
            '--noali',
            str(HMM_DB),
            str(faa),
        ]
        print(f'  {s}: running hmmsearch...')
        print('  CMD:', ' '.join(cmd))
        subprocess.run(cmd, check=True, capture_output=True)
    else:
        print(f'  {s}: cached tblout found')

    # Parse tblout
    hits = []
    with open(tblout) as f:
        for line in f:
            if line.startswith('#') or not line.strip():
                continue
            cols  = line.split()
            hits.append({
                'protein':  cols[0],
                'profile':  cols[2],
                'evalue':   float(cols[4]),
                'score':    float(cols[5]),
            })
    results[s] = hits

# ── Step 3: Summary table ────────────────────────────────────────────────────
print('\n=== KaiABC screen results ===')
print(f'{"Sample":<25} {"KaiA":>6} {"KaiB":>6} {"KaiC":>6}  {"Hits"}')
print('-' * 65)

any_hits = False
for s in ALL_SAMPLES:
    hits = results[s]
    kai_a = sum(1 for h in hits if 'KaiA' in h['profile'])
    kai_b = sum(1 for h in hits if 'KaiB' in h['profile'] or 'circ_KaiB' in h['profile'])
    kai_c = sum(1 for h in hits if 'KaiC' in h['profile'] or 'circ_KaiC' in h['profile'])
    flag  = ' ◄ HIT' if hits else ''
    print(f'{s:<25} {kai_a:>6} {kai_b:>6} {kai_c:>6}{flag}')
    if hits:
        any_hits = True
        for h in hits:
            print(f'  → {h["profile"]:15s}  E={h["evalue"]:.1e}  score={h["score"]:.1f}  protein={h["protein"]}')

if not any_hits:
    print('\nNo KaiABC hits detected across any sample (E ≤ 1e-5).')
    print('These cyanobacteria lack the canonical circadian clock.')


=== Step 1: Prodigal gene prediction ===
  KY-Mam-100624-B: already done (123,342 proteins)
  KY-Mam-100624-C: already done (75,230 proteins)
  MN-Por-091024-A: already done (55,760 proteins)
  TN-Cla-092324-E: already done (11,288 proteins)
  WI-Bar-090824-B: already done (74,797 proteins)
  NC-Che-092424-E: already done (126,869 proteins)
  GA-Cor-092624-B: already done (42,143 proteins)
  FL-Wim-100124-B: already done (160,489 proteins)
  MN-Col-091924-B: already done (86,287 proteins)
  FL-Hom-052325-G: already done (6,016 proteins)

=== Step 2: hmmsearch vs KaiABC_correct.hmm ===
  KY-Mam-100624-B: cached tblout found
  KY-Mam-100624-C: cached tblout found
  MN-Por-091024-A: cached tblout found
  TN-Cla-092324-E: cached tblout found
  WI-Bar-090824-B: cached tblout found
  NC-Che-092424-E: cached tblout found
  GA-Cor-092624-B: cached tblout found
  FL-Wim-100124-B: cached tblout found
  MN-Col-091924-B: cached tblout found
  FL-Hom-052325-G: cached tblout found

=== KaiABC screen

In [15]:
# ── 7b. Trace KaiABC hits to source organism (Kraken2 taxonomy) ──────────────
# Prodigal names proteins as {contig_id}_{gene_n}, so stripping the suffix
# gives the source contig. Kraken2 per-contig TSV maps each contig to a taxon.

import re, csv
from collections import defaultdict

KAI_DIR = PROJECT / 'results' / '14_kai_screen'
EVALUE_STRONG = 1e-10   # threshold for "confident" hits

# Profiles that define a complete canonical locus
KAI_PROFILES = {
    'KaiA':      {'KaiA', 'KaiA_N'},
    'KaiB':      {'circ_KaiB'},
    'KaiC':      {'circ_KaiC'},
}

def contig_of(protein_id):
    """contig_283_297  →  contig_283"""
    return '_'.join(protein_id.split('_')[:2])

def load_kraken_index(sample):
    """Return dict: contig_id → taxonomy string."""
    tsv = TAX_DIR / sample / 'contigs.kraken.tsv'
    idx = {}
    if not tsv.exists():
        return idx
    with open(tsv) as f:
        for line in f:
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 3:
                continue
            contig = parts[1]
            taxon  = parts[2]   # e.g. "Limnothrix redekei (taxid 1296593)"
            idx[contig] = taxon
    return idx

def load_hits(tblout, evalue_cutoff=EVALUE_STRONG):
    hits = []
    with open(tblout) as f:
        for line in f:
            if line.startswith('#') or not line.strip():
                continue
            cols = line.split()
            ev = float(cols[4])
            if ev <= evalue_cutoff:
                hits.append({'protein': cols[0], 'profile': cols[2], 'evalue': ev, 'score': float(cols[5])})
    return hits

# ── Collect per-sample results ───────────────────────────────────────────────
rows = []

for s in ALL_SAMPLES:
    tblout = KAI_DIR / f'{s}_kai.tblout'
    if not tblout.exists():
        continue

    hits   = load_hits(tblout, evalue_cutoff=EVALUE_STRONG)
    if not hits:
        continue

    kraken = load_kraken_index(s)

    # Group hits by contig
    by_contig = defaultdict(list)
    for h in hits:
        by_contig[contig_of(h['protein'])].append(h)

    for contig, chits in by_contig.items():
        profiles_hit = set()
        for h in chits:
            for gene, prof_set in KAI_PROFILES.items():
                if h['profile'] in prof_set:
                    profiles_hit.add(gene)

        has_A = 'KaiA' in profiles_hit
        has_B = 'KaiB' in profiles_hit
        has_C = 'KaiC' in profiles_hit
        complete = has_A and has_B and has_C

        taxon = kraken.get(contig, 'unclassified')
        best_e = min(h['evalue'] for h in chits)

        rows.append({
            'sample':   s,
            'contig':   contig,
            'taxon':    taxon,
            'KaiA':     '✓' if has_A else '—',
            'KaiB':     '✓' if has_B else '—',
            'KaiC':     '✓' if has_C else '—',
            'complete': complete,
            'best_e':   best_e,
        })

# Sort: complete loci first, then by sample
rows.sort(key=lambda r: (not r['complete'], r['sample']))

# ── Display ──────────────────────────────────────────────────────────────────
import pandas as pd

df = pd.DataFrame(rows)[['sample','contig','KaiA','KaiB','KaiC','complete','best_e','taxon']]
df['taxon_short'] = df['taxon'].str.replace(r'\s*\(taxid \d+\)', '', regex=True).str[:60]

print(f"{'Sample':<25} {'Contig':<18} {'A':>3} {'B':>3} {'C':>3} {'Complete':>8}  {'Best E':>10}  Taxon")
print('-' * 110)
for _, r in df.iterrows():
    flag = ' ◄ COMPLETE' if r['complete'] else ''
    print(f"{r['sample']:<25} {r['contig']:<18} {r['KaiA']:>3} {r['KaiB']:>3} {r['KaiC']:>3} {str(r['complete']):>8}  {r['best_e']:>10.1e}  {r['taxon_short']}{flag}")

n_complete = df['complete'].sum()
print(f"\n{n_complete} complete KaiABC loci (A+B+C on same contig, E ≤ {EVALUE_STRONG:.0e})")
print(f"{len(df) - n_complete} partial / lone KaiC hits (may be other ATPases or distant homologs)")


Sample                    Contig               A   B   C Complete      Best E  Taxon
--------------------------------------------------------------------------------------------------------------
GA-Cor-092624-B           contig_105           ✓   ✓   ✓     True    2.7e-311  Synechococcus sp. PCC 6312 ◄ COMPLETE
MN-Col-091924-B           contig_3535          ✓   ✓   ✓     True     0.0e+00  Synechococcus elongatus PCC 7942 ◄ COMPLETE
MN-Por-091024-A           contig_283           ✓   ✓   ✓     True    6.4e-308  Nostoc sp. 'Peltigera membranacea cyanobiont' N6 ◄ COMPLETE
MN-Por-091024-A           contig_360           ✓   ✓   ✓     True    8.2e-313  Cyanobium gracile PCC 6307 ◄ COMPLETE
TN-Cla-092324-E           contig_35            ✓   ✓   ✓     True    7.3e-312  Synechococcus sp. PCC 6312 ◄ COMPLETE
FL-Hom-052325-G           contig_46            ✓   —   —    False     4.8e-50  Nostocales cyanobacterium HT-58-2
FL-Hom-052325-G           contig_406           —   ✓   ✓    False     2.1e-83 

In [ ]:
# ── 7c. KaiABC dot matrix — slide figure ─────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from collections import defaultdict

KAI_DIR      = PROJECT / 'results' / '14_kai_screen'
EVALUE_STRONG = 1e-10   # threshold for complete-locus calls

KAI_PROFILES = {
    'KaiA':     {'KaiA', 'KaiA_N'},
    'KaiB':     {'circ_KaiB'},
    'KaiB-long': {'KaiB_longform_Anabaena7120'},
    'KaiC':     {'circ_KaiC'},
}

# ── Sample display labels + short taxonomy (fill after kai-trace-code runs) ──
SAMPLE_META = {
    'MN-Por-091024-A':  ('Portage Lake, MN',        'Cyanobium + Calothrix'),
    'TN-Cla-092324-E':  ('Clarksville, TN',          'Limnothrix / Planktothrix'),
    'GA-Cor-092624-B':  ('Cordele, GA',              'Limnothrix'),
    'MN-Col-091924-B':  ('Columbia Lake, MN ★',      'Synechococcus + chloroplast contigs'),
    'FL-Hom-052325-G':  ('Homosassa, FL',            'Nodosilinea'),
    'KY-Mam-100624-B':  ('Mammoth Cave, KY (surface)','Loriellopsis'),
    'KY-Mam-100624-C':  ('Mammoth Cave, KY (interior) ★', 'Chloroplast contigs — no cyano ID'),
    'FL-Wim-100124-B':  ('Wimberly Lake, FL',        'No cyano confirmed'),
    'NC-Che-092424-E':  ('Cheraw, NC',               'No cyano confirmed'),
    'WI-Bar-090824-B':  ('Barfield Lake, WI',        'No cyano confirmed'),
}
# ★ = samples carrying the chloroplast-bin contigs examined in this notebook

# Row order: complete loci → partial → absent
ROW_ORDER = [
    'MN-Por-091024-A', 'TN-Cla-092324-E', 'GA-Cor-092624-B', 'MN-Col-091924-B',
    'FL-Hom-052325-G', 'KY-Mam-100624-B',
    'KY-Mam-100624-C', 'FL-Wim-100124-B', 'NC-Che-092424-E', 'WI-Bar-090824-B',
]
GENES = ['KaiA', 'KaiB', 'KaiB-long', 'KaiC']

# ── Parse tblout files ────────────────────────────────────────────────────────
def parse_tblout(path, evalue_cutoff):
    hits = defaultdict(set)   # contig → set of genes
    with open(path) as f:
        for line in f:
            if line.startswith('#') or not line.strip():
                continue
            cols = line.split()
            ev   = float(cols[4])
            if ev > evalue_cutoff:
                continue
            contig = '_'.join(cols[0].split('_')[:2])
            for gene, profs in KAI_PROFILES.items():
                if cols[2] in profs:
                    hits[contig].add(gene)
    return hits

# For each sample: which genes detected (on any contig), and is there a complete locus?
sample_genes   = {}   # sample → set of genes with strong hits
sample_complete = {}  # sample → bool

for s in ALL_SAMPLES:
    tblout = KAI_DIR / f'{s}_kai.tblout'
    if not tblout.exists():
        sample_genes[s]    = set()
        sample_complete[s] = False
        continue
    contig_hits = parse_tblout(tblout, EVALUE_STRONG)
    all_genes   = set().union(*contig_hits.values()) if contig_hits else set()
    complete    = any(
        ({'KaiA', 'KaiC'} | ({'KaiB'} if 'KaiB' in genes else {'KaiB-long'})).issubset(genes)
        for genes in contig_hits.values()
    )
    sample_genes[s]    = all_genes
    sample_complete[s] = complete

# ── Build matrix ─────────────────────────────────────────────────────────────
BG, AX_BG = '#2b2b2b', '#2b2b2b'
COL_COMPLETE = "#20AD62"   # green  — complete locus
COL_PARTIAL  = '#5C85D6'   # blue  — gene present, locus incomplete
COL_ABSENT   = '#444444'   # dark grey — absent
COL_NOVEL    = '#E8824E'   # orange highlight for novel-sample rows

n_rows = len(ROW_ORDER)
n_cols = len(GENES)

with plt.style.context('dark_background'):
    fig, ax = plt.subplots(figsize=(8.5, 6), facecolor=BG)
    ax.set_facecolor(AX_BG)

    for yi, s in enumerate(ROW_ORDER):
        row_y   = n_rows - yi - 1
        complete = sample_complete[s]
        genes    = sample_genes[s]
        is_plastid = '★' in SAMPLE_META[s][0]

        # Row background stripe for novel samples
        if is_plastid:
            ax.axhspan(row_y - 0.45, row_y + 0.45, color='#3a2a1a', zorder=0)

        for xi, gene in enumerate(GENES):
            present = gene in genes
            if present and complete:
                color, size, marker = COL_COMPLETE, 220, 'o'
            elif present:
                color, size, marker = COL_PARTIAL, 150, 'o'
            else:
                color, size, marker = COL_ABSENT, 60, 'o'

            ax.scatter(xi, row_y, s=size, color=color,
                       zorder=3, linewidths=0.5,
                       edgecolors='#888' if not present else 'none')

    # ── Axes formatting ───────────────────────────────────────────────────────
    ax.set_xticks(range(n_cols))
    ax.set_xticklabels(GENES, fontsize=12, fontweight='bold')
    ax.set_yticks(range(n_rows))
    ylabels = []
    for s in reversed(ROW_ORDER):
        loc, taxon = SAMPLE_META[s]
        ylabels.append(f'{loc}\n{taxon}')
    ax.set_yticklabels(ylabels, fontsize=8)
    ax.tick_params(axis='x', bottom=False, top=True, labelbottom=False, labeltop=True)
    ax.xaxis.set_label_position('top')

    # Divider between complete/partial and absent sections
    ax.axhline(n_rows - 6 - 0.5, color='#666', lw=0.8, ls='--')

    ax.set_xlim(-0.6, n_cols - 0.4)
    ax.set_ylim(-0.6, n_rows - 0.4)
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.grid(axis='x', color='#444', lw=0.5, zorder=1)

    # ── Legend ────────────────────────────────────────────────────────────────
    handles = [
        mpatches.Patch(color=COL_COMPLETE, label='Complete locus (A+B+C on same contig)'),
        mpatches.Patch(color=COL_PARTIAL,  label='Gene detected (locus incomplete)'),
        mpatches.Patch(color=COL_ABSENT,   label='Not detected'),
        mpatches.Patch(color='#3a2a1a',    label='★ Sample contains chloroplast contigs'),
    ]
    fig.legend(handles=handles, fontsize=7.5, loc='lower center',
               ncol=2, facecolor='#333', edgecolor='#555', framealpha=0.9,
               bbox_to_anchor=(0.5, 0.0))

    ax.set_title('KaiABC circadian clock gene screen\nacross 10 freshwater metagenomes',
                 fontsize=11, fontweight='bold', pad=28)

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.18)
    fig.savefig(FIG_DIR / 'kaiABC_dotmatrix.pdf', bbox_inches='tight', dpi=200, facecolor=BG)
    fig.savefig(FIG_DIR / 'kaiABC_dotmatrix.png', bbox_inches='tight', dpi=200, facecolor=BG)
    plt.show()
    print(f'Saved → {FIG_DIR}/kaiABC_dotmatrix.pdf')


# 7D. Permissive read mapping to kaiC
Step 1 — Build a reference panel from the KaiC sequences already confirmed in this dataset (MN-Por, TN-Cla, GA-Cor, MN-Col). These represent the actual freshwater KaiC diversity in your samples — a better reference than downloading canonical lab strains, because they're already tuned to similar environments.

Step 2 — Map with minimap2 map-ont with --secondary=yes -N 10 so reads that partially match any reference still get reported. No identity cutoff is applied — minimap2 will recruit whatever it can align.

Step 3 — samtools coverage reports per-reference: number of reads mapped, bases covered, mean depth, and coverage fraction. The interpretation block at the bottom gives you a plain-English result either way.

What the result means:

Zero coverage → strengthens absence claim; add "no reads recruited at map-ont sensitivity across freshwater KaiC diversity"
Any coverage → follow up by checking the identity of mapped reads (samtools view + cigar inspection) to determine if it's true signal or spurious low-identity noise

In [17]:
# ── 7d. Permissive read mapping to KaiC — absence test for KY-Mam-C ──────────
# Strategy:
#   1. Extract KaiC CDS sequences from all samples with confirmed hits
#      (coordinates embedded in Prodigal protein headers in tblout)
#   2. Map KY-Mam-C filtered reads against this reference at relaxed identity
#   3. Check coverage — even divergent homologs should recruit reads if present
#
# Interpretation:
#   Coverage > 0 at low identity  → KaiC-like sequence present but divergent
#   Coverage = 0 across all refs  → stronger (but not definitive) absence signal

import re
from Bio import SeqIO

KAIC_DIR  = PROJECT / 'results' / '14_kai_screen' / 'kaiC_refs'
KAIC_DIR.mkdir(parents=True, exist_ok=True)

READS     = PROJECT / 'results' / '02_filter' / 'KY-Mam-100624-C.flt.fq.gz'
KAIC_REF  = KAIC_DIR / 'kaiC_refs.fna'
BAM_OUT   = KAIC_DIR / 'KY-Mam-C_vs_kaiC.bam'
DEPTH_OUT = KAIC_DIR / 'KY-Mam-C_vs_kaiC.depth'

EVALUE_STRONG = 1e-10
KAIC_PROF = {'circ_KaiC'}

# ── Step 1: Extract KaiC CDS sequences from confirmed-positive samples ────────
print('=== Step 1: Extracting KaiC reference sequences ===')

# Samples with complete or strong KaiC hits
KAIC_SOURCES = [
    'MN-Por-091024-A', 'TN-Cla-092324-E',
    'GA-Cor-092624-B', 'MN-Col-091924-B',
]

ref_seqs = []

for s in KAIC_SOURCES:
    tblout   = KAI_DIR / f'{s}_kai.tblout'
    assembly = POLISH_DIR / f'{s}.racon2.fasta'
    if not tblout.exists() or not assembly.exists():
        print(f'  {s}: missing tblout or assembly — skip')
        continue

    # Load assembly into dict
    asm = {r.id: r for r in SeqIO.parse(assembly, 'fasta')}

    with open(tblout) as f:
        for line in f:
            if line.startswith('#') or not line.strip():
                continue
            cols = line.split()
            if float(cols[4]) > EVALUE_STRONG:
                continue
            if cols[2] not in KAIC_PROF:
                continue

            protein_id = cols[0]
            # Prodigal encodes coords in description: # start # end # strand
            m = re.search(r'# (\d+) # (\d+) # (-?1)', line)
            if not m:
                continue
            start, end, strand = int(m.group(1))-1, int(m.group(2)), int(m.group(3))

            contig_id = '_'.join(protein_id.split('_')[:2])
            if contig_id not in asm:
                print(f'  {s}: contig {contig_id} not in assembly — skip')
                continue

            seq = asm[contig_id].seq[start:end]
            if strand == -1:
                seq = seq.reverse_complement()

            ref_id = f'{s}__{protein_id}'
            from Bio.SeqRecord import SeqRecord
            ref_seqs.append(SeqRecord(seq, id=ref_id, description=''))
            print(f'  {s}: extracted {protein_id} ({len(seq)} bp, E={float(cols[4]):.1e})')

SeqIO.write(ref_seqs, KAIC_REF, 'fasta')
print(f'\n{len(ref_seqs)} KaiC reference sequences → {KAIC_REF}')

# ── Step 2: Map KY-Mam-C reads (relaxed — no identity filter) ────────────────
print('\n=== Step 2: Mapping KY-Mam-C reads to KaiC references ===')

if not BAM_OUT.exists():
    # map-ont preset; no --min-cnt filter; allow secondary alignments
    cmd_map = [
        'minimap2',
        '-ax', 'map-ont',
        '--secondary=yes',
        '-N', '10',          # keep up to 10 secondary alignments per read
        '-t', '4',
        str(KAIC_REF),
        str(READS),
    ]
    cmd_sort = ['samtools', 'sort', '-o', str(BAM_OUT)]
    print('CMD (map):', ' '.join(cmd_map))
    print('CMD (sort): piped to samtools sort')

    mm2 = subprocess.run(cmd_map, capture_output=True)
    subprocess.run(cmd_sort, input=mm2.stdout, check=True)
    subprocess.run(['samtools', 'index', str(BAM_OUT)], check=True)
    print('Mapping complete.')
else:
    print(f'Cached BAM found: {BAM_OUT}')

# ── Step 3: Coverage per reference ───────────────────────────────────────────
print('\n=== Step 3: Coverage per KaiC reference ===')

depth_res = subprocess.run(
    ['samtools', 'coverage', str(BAM_OUT)],
    capture_output=True, text=True, check=True
)
print(depth_res.stdout)

# Also get mean identity of mapped reads
flagstat = subprocess.run(
    ['samtools', 'flagstat', str(BAM_OUT)],
    capture_output=True, text=True, check=True
)
print(flagstat.stdout)

# ── Interpretation ────────────────────────────────────────────────────────────
import pandas as pd, io

cov_df = pd.read_csv(io.StringIO(depth_res.stdout), sep='\t')
total_reads = 161253   # from flagstat — update if re-run on different sample

# Display explicit per-reference summary
display_cols = ['#rname', 'endpos', 'numreads', 'covbases', 'coverage', 'meandepth', 'meanmapq']
print('\n=== Per-reference coverage summary ===')
print(cov_df[display_cols].to_string(index=False))

total_mapped = cov_df['numreads'].sum()
pct_mapped   = total_mapped / total_reads * 100

print(f'\nTotal reads:  {total_reads:,}')
print(f'Reads mapped: {int(total_mapped):,}  ({pct_mapped:.4f}%)')

# Threshold: require ≥5 reads AND ≥10% coverage of reference to call presence
MIN_READS = 5
MIN_COV   = 10.0
hits = cov_df[(cov_df['numreads'] >= MIN_READS) & (cov_df['coverage'] >= MIN_COV)]

print(f'\nRefs with ≥{MIN_READS} reads AND ≥{MIN_COV}% coverage: {len(hits)}')

if hits.empty:
    print('\n>>> RESULT: No meaningful KaiC coverage detected.')
    print(f'    Only {int(total_mapped)} read(s) mapped out of {total_reads:,} ({pct_mapped:.4f}%).')
    print('    This is consistent with KaiC absence in KY-Mam-C.')
    print('    Caveat: genome completeness <2% — true absence cannot be confirmed.')
else:
    print('\n>>> RESULT: KaiC-like reads detected:')
    print(hits[display_cols].to_string(index=False))
    print('    Investigate mapped reads for identity before concluding presence.')


=== Step 1: Extracting KaiC reference sequences ===
  MN-Por-091024-A: extracted contig_360_42 (1545 bp, E=8.2e-313)
  MN-Por-091024-A: extracted contig_283_299 (1560 bp, E=6.4e-308)
  MN-Por-091024-A: extracted contig_219_325 (1695 bp, E=9.0e-107)
  MN-Por-091024-A: extracted contig_1029_14 (1458 bp, E=1.5e-42)
  TN-Cla-092324-E: extracted contig_35_3420 (1515 bp, E=7.3e-312)
  GA-Cor-092624-B: extracted contig_105_109 (1515 bp, E=2.7e-311)
  MN-Col-091924-B: extracted contig_3535_650 (1560 bp, E=0.0e+00)

7 KaiC reference sequences → /Users/jaemac/Desktop/Palmer_Lab/Cyanobacteria/GenomicAnalyses/MetagenomicAnalyses/metagenome/results/14_kai_screen/kaiC_refs/kaiC_refs.fna

=== Step 2: Mapping KY-Mam-C reads to KaiC references ===
Cached BAM found: /Users/jaemac/Desktop/Palmer_Lab/Cyanobacteria/GenomicAnalyses/MetagenomicAnalyses/metagenome/results/14_kai_screen/kaiC_refs/KY-Mam-C_vs_kaiC.bam

=== Step 3: Coverage per KaiC reference ===
#rname	startpos	endpos	numreads	covbases	coverage

In [ ]:
# ────── 7e. Resolve open question: which organism carries KaiABC in MN-Col? ──────────
# contig_3535 carries the complete KaiABC locus in MN-Col-091924-B.
# MN-Col also hosts a novel Oscillatoriales clade (contig_1504).
# This cell confirms which organism contig_3535 belongs to.

MN_COL = 'MN-Col-091924-B'
KAIC_CONTIG = 'contig_3535'

# ────── 1. Kraken2 classification ────────────────────────────────────────────────────────────────
kraken_tsv = TAX_DIR / MN_COL / 'contigs.kraken.tsv'
kraken_hit = None
with open(kraken_tsv) as f:
    for line in f:
        parts = line.rstrip('\n').split('\t')
        if parts[1] == KAIC_CONTIG:
            kraken_hit = parts[2]
            break

print(f'Kraken2 classification of {KAIC_CONTIG}:')
print(f'  {kraken_hit}')

# ────── 2. Is contig_3535 in the novel cyano contig list? ───────────────────────────────────────
plastid_fna = OUT_DIR / 'query_16S.fna'
mn_plastid_set = set()
if plastid_fna.exists():
    with open(plastid_fna) as f:
        for line in f:
            if line.startswith('>') and MN_COL.replace('091924-B','B') in line or 'MN-Col' in line:
                mn_plastid_set.add(line.strip())

print(f'\nChloroplast-bin contigs for MN-Col (from this notebook):')
if mn_plastid_set:
    for c in mn_plastid_set:
        print(f'  {c}')
else:
    # fallback: grep the alignment file
    import subprocess
    res = subprocess.run(['grep', '>', str(plastid_fna)], capture_output=True, text=True)
    mn_plastid = [l.strip() for l in res.stdout.splitlines() if 'MN-Col' in l or 'MN-Col-B' in l]
    for c in mn_plastid:
        print(f'  {c}')

mn_plastid_all = mn_plastid_set if mn_plastid_set else set(mn_plastid) if 'mn_plastid' in dir() else set()
print(f'\n  {KAIC_CONTIG} in chloroplast-bin list: {any(KAIC_CONTIG in c for c in mn_plastid_all)}')

# ────── 3. SILVA 16S identity for contig_3535 ───────────────────────────────────────────────────
silva_tsv = TAX_DIR / MN_COL / 'blast_16S_silva.tsv'
print(f'\nSILVA 16S BLAST hits on {KAIC_CONTIG}:')
with open(silva_tsv) as f:
    for line in f:
        if KAIC_CONTIG in line:
            cols = line.strip().split('\t')
            # cols: qseqid, sseqid, stitle, pident, length, evalue
            print(f'  {cols[2].split(";")[-1].strip()}  — {cols[3]}% identity')
            break

# ────── Conclusion ───────────────────────────────────────────────────────────────────────────────
print("""
=== CONCLUSION ===
contig_3535 (KaiABC locus, MN-Col) is Synechococcus elongatus — a genuine cyanobacterium.
contig_1504 carries no KaiABC signal because it is a green algal chloroplast genome, and
plastids do not carry kaiABC at all (lost during endosymbiotic genome reduction).

NOTE (FLAG 4): the cross-site 'same novel lineage, no clock' reading below does not
hold — both sites' contigs are plastids, so their lack of a clock operon is expected and
carries no information about cave adaptation. Original text retained:
This closes the open question: KaiABC absence is consistent across the novel
Oscillatoriales genus at both sites (KY-Mam-C cave interior + MN-Col Columbia Lake).
Two geographically separated environments, same novel lineage, no clock detected.
""")
